In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from PIL import Image
import numpy as np
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
from zipfile import ZipFile
import glob
import random
import shutil
from pathlib import Path
import cv2

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [ ]:
class Config:
    """Configuration for Pix2Pix training"""

    # Paths
    images_zip = 'images.zip'  # RGB fundus images
    masks_zip = 'masks.zip'    # RGB neovascularization masks
    output_dir = 'generated_images'
    checkpoint_dir = 'checkpoints'

    # Image parameters
    img_size = 512  # 512x512 resolution
    img_channels = 3  # RGB images
    mask_channels = 3  # RGB masks

    # Training parameters
    batch_size = 2  # Reduced for 512x512 (increase to 4 if you have enough GPU memory)
    num_epochs = 200
    lr = 0.0002
    beta1 = 0.5
    beta2 = 0.999

    # Loss weights
    lambda_l1 = 100  # L1 loss weight (structure preservation)
    lambda_perceptual = 0.1  # Perceptual loss weight

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Generation
    num_generate = 100  # Number of new synthetic images to generate

    # Save frequency
    save_checkpoint_every = 20  # Save checkpoint every N epochs
    visualize_every = 10  # Visualize results every N epochs

config = Config()

In [ ]:
# Check for nested 'images' folder first
nested_images = os.path.join(images, 'images')
if os.path.exists(nested_images) and os.path.isdir(nested_images):
    actual_images_dir = nested_images  # Use images/images/

# Check for nested 'masks' folder first
nested_masks = os.path.join(masks, 'masks')
if os.path.exists(nested_masks) and os.path.isdir(nested_masks):
    actual_masks_dir = nested_masks    # Use masks/masks/

NameError: name 'images' is not defined

In [ ]:
class RetinalDataset(Dataset):
    """
    Dataset for RGB fundus images and RGB neovascularization masks
    Input: RGB fundus image (3 channels)
    Condition: RGB neovascularization mask (3 channels)
    Output: RGB fundus image with neovascular vessels
    """

    def __init__(self, image_dir, mask_dir, img_size=512):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        # Get all image files (common extensions)
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        self.image_files = []
        for ext in image_extensions:
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext)))
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext.upper())))
        self.image_files = sorted(self.image_files)

        # Get all mask files
        mask_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        self.mask_files = []
        for ext in mask_extensions:
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext)))
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext.upper())))
        self.mask_files = sorted(self.mask_files)

        # Create dictionary for fast lookup
        mask_dict = {}
        for mask_path in self.mask_files:
            mask_basename = os.path.splitext(os.path.basename(mask_path))[0]
            mask_dict[mask_basename] = mask_path

        # Match images with masks by basename (extension-agnostic)
        valid_pairs = []
        unmatched_images = []

        for img_path in self.image_files:
            img_basename = os.path.splitext(os.path.basename(img_path))[0]
            if img_basename in mask_dict:
                valid_pairs.append((img_path, mask_dict[img_basename]))
            else:
                unmatched_images.append(img_basename)

        self.pairs = valid_pairs

        print(f"\n{'='*60}")
        print(f"Dataset Statistics:")
        print(f"{'='*60}")
        print(f"✓ Found {len(self.pairs)} valid image-mask pairs")
        print(f"  Image size: {img_size}x{img_size}")
        print(f"  Image channels: 3 (RGB)")
        print(f"  Mask channels: 3 (RGB)")

        if len(unmatched_images) > 0:
            print(f"\n⚠️ Warning: {len(unmatched_images)} images without matching masks:")
            for name in unmatched_images[:5]:
                print(f"     - {name}")
            if len(unmatched_images) > 5:
                print(f"     ... and {len(unmatched_images) - 5} more")

        if len(self.pairs) > 0:
            img_file = os.path.basename(self.pairs[0][0])
            mask_file = os.path.basename(self.pairs[0][1])
            print(f"\nExample pair:")
            print(f"  Image: {img_file}")
            print(f"  Mask:  {mask_file}")
        print(f"{'='*60}\n")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # Load RGB images (both fundus and mask are RGB)
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('RGB')  # RGB mask

        # Resize
        image = image.resize((self.img_size, self.img_size), Image.BICUBIC)
        mask = mask.resize((self.img_size, self.img_size), Image.BICUBIC)

        # Convert to tensors
        image = transforms.ToTensor()(image)  # [3, H, W]
        mask = transforms.ToTensor()(mask)    # [3, H, W]

        # Normalize to [-1, 1]
        image = image * 2 - 1
        mask = mask * 2 - 1

        return mask, image  # (condition, target)


In [ ]:
class UNetBlock(nn.Module):
    """U-Net encoder-decoder block"""

    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()

        if down:
            # Encoder
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, inplace=True)
            )
        else:
            # Decoder
            layers = [
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            if use_dropout:
                layers.append(nn.Dropout(0.5))
            self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.conv(x)

class Generator(nn.Module):
    """
    U-Net Generator for Pix2Pix
    Input: RGB neovascularization mask (3 channels)
    Output: RGB fundus image with neovascular vessels (3 channels)
    """

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        # Encoder (downsampling)
        self.down1 = nn.Conv2d(in_channels, 64, 4, 2, 1)  # 512 -> 256
        self.down2 = UNetBlock(64, 128, down=True)        # 256 -> 128
        self.down3 = UNetBlock(128, 256, down=True)       # 128 -> 64
        self.down4 = UNetBlock(256, 512, down=True)       # 64 -> 32
        self.down5 = UNetBlock(512, 512, down=True)       # 32 -> 16
        self.down6 = UNetBlock(512, 512, down=True)       # 16 -> 8
        self.down7 = UNetBlock(512, 512, down=True)       # 8 -> 4
        self.down8 = UNetBlock(512, 512, down=True)       # 4 -> 2

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, 4, 2, 1),                 # 2 -> 1
            nn.ReLU(inplace=True)
        )

        # Decoder (upsampling)
        self.up1 = UNetBlock(512, 512, down=False, use_dropout=True)      # 1 -> 2
        self.up2 = UNetBlock(1024, 512, down=False, use_dropout=True)     # 2 -> 4
        self.up3 = UNetBlock(1024, 512, down=False, use_dropout=True)     # 4 -> 8
        self.up4 = UNetBlock(1024, 512, down=False)                       # 8 -> 16
        self.up5 = UNetBlock(1024, 512, down=False)                       # 16 -> 32
        self.up6 = UNetBlock(1024, 256, down=False)                       # 32 -> 64
        self.up7 = UNetBlock(512, 128, down=False)                        # 64 -> 128
        self.up8 = UNetBlock(256, 64, down=False)                         # 128 -> 256

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),               # 256 -> 512
            nn.Tanh()
        )

    def forward(self, x):
        # Encoder with skip connections
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        # Bottleneck
        bottleneck = self.bottleneck(d8)

        # Decoder with skip connections
        u1 = self.up1(bottleneck)
        u2 = self.up2(torch.cat([u1, d8], dim=1))
        u3 = self.up3(torch.cat([u2, d7], dim=1))
        u4 = self.up4(torch.cat([u3, d6], dim=1))
        u5 = self.up5(torch.cat([u4, d5], dim=1))
        u6 = self.up6(torch.cat([u5, d4], dim=1))
        u7 = self.up7(torch.cat([u6, d3], dim=1))
        u8 = self.up8(torch.cat([u7, d2], dim=1))

        return self.final(torch.cat([u8, d1], dim=1))


In [ ]:
class Discriminator(nn.Module):
    """
    PatchGAN Discriminator
    Input: Concatenated RGB mask (3) + RGB image (3) = 6 channels
    """

    def __init__(self, in_channels=6):  # mask (3) + image (3)
        super().__init__()

        self.model = nn.Sequential(
            # Input: 512x512
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            # 256x256
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 128x128
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # 64x64
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32
            nn.Conv2d(512, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # 31x31
            nn.Conv2d(512, 1, 4, 1, 1)
            # Output: 30x30 patch predictions
        )

    def forward(self, mask, image):
        x = torch.cat([mask, image], dim=1)  # Concatenate along channel dimension
        return self.model(x)


In [ ]:
def save_sample_images(mask, real, fake, epoch, batch_idx, save_dir='samples'):
    """Save sample images during training"""
    os.makedirs(save_dir, exist_ok=True)

    # Denormalize from [-1, 1] to [0, 1]
    mask = (mask + 1) / 2
    real = (real + 1) / 2
    fake = (fake + 1) / 2

    # Create grid
    comparison = torch.cat([mask[:4], real[:4], fake[:4]], dim=0)
    grid = make_grid(comparison, nrow=4, padding=2, normalize=False)

    # Save
    save_path = os.path.join(save_dir, f'epoch_{epoch}_batch_{batch_idx}.png')
    save_image(grid, save_path)

def visualize_progress(mask, real_image, fake_image, epoch, save_path='progress.png'):
    """Visualize training progress"""

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    num_samples = min(4, mask.size(0))

    for i in range(num_samples):
        # Denormalize
        mask_np = (mask[i].cpu().permute(1, 2, 0).numpy() + 1) / 2
        real_np = (real_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2
        fake_np = (fake_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2

        # Clip to valid range
        mask_np = np.clip(mask_np, 0, 1)
        real_np = np.clip(real_np, 0, 1)
        fake_np = np.clip(fake_np, 0, 1)

        # Plot
        axes[0, i].imshow(mask_np)
        axes[0, i].set_title(f'Neovascular Mask {i+1}', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(real_np)
        axes[1, i].set_title(f'Real Fundus {i+1}', fontsize=10)
        axes[1, i].axis('off')

        axes[2, i].imshow(fake_np)
        axes[2, i].set_title(f'Generated Fundus {i+1}', fontsize=10)
        axes[2, i].axis('off')

    plt.suptitle(f'Training Progress - Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def train_pix2pix(dataloader, config):
    """Train Pix2Pix GAN"""

    print("\n" + "="*60)
    print("INITIALIZING TRAINING")
    print("="*60)

    # Initialize models
    generator = Generator(in_channels=3, out_channels=3).to(config.device)
    discriminator = Discriminator(in_channels=6).to(config.device)

    # Count parameters
    gen_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
    disc_params = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
    print(f"\nModel Architecture:")
    print(f"  Generator parameters: {gen_params:,}")
    print(f"  Discriminator parameters: {disc_params:,}")

    # Optimizers
    opt_gen = optim.Adam(generator.parameters(), lr=config.lr, betas=(config.beta1, config.beta2))
    opt_disc = optim.Adam(discriminator.parameters(), lr=config.lr, betas=(config.beta1, config.beta2))

    # Loss functions
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_l1 = nn.L1Loss()

    # Create directories
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    os.makedirs('samples', exist_ok=True)

    print(f"\nTraining Configuration:")
    print(f"  Device: {config.device}")
    print(f"  Epochs: {config.num_epochs}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Learning rate: {config.lr}")
    print(f"  Image size: {config.img_size}x{config.img_size}")
    print(f"  L1 weight: {config.lambda_l1}")
    print("="*60 + "\n")

    # Training loop
    for epoch in range(config.num_epochs):
        generator.train()
        discriminator.train()

        loop = tqdm(dataloader, desc=f'Epoch {epoch+1}/{config.num_epochs}')

        epoch_g_loss = 0
        epoch_d_loss = 0
        epoch_l1_loss = 0

        for batch_idx, (mask, real_image) in enumerate(loop):
            mask = mask.to(config.device)
            real_image = real_image.to(config.device)
            batch_size = mask.size(0)

            # ==================
            # Train Discriminator
            # ==================
            opt_disc.zero_grad()

            # Generate fake images
            fake_image = generator(mask)

            # Real images
            real_pred = discriminator(mask, real_image)
            real_labels = torch.ones_like(real_pred)
            loss_real = criterion_gan(real_pred, real_labels)

            # Fake images
            fake_pred = discriminator(mask, fake_image.detach())
            fake_labels = torch.zeros_like(fake_pred)
            loss_fake = criterion_gan(fake_pred, fake_labels)

            # Total discriminator loss
            loss_disc = (loss_real + loss_fake) * 0.5
            loss_disc.backward()
            opt_disc.step()

            # ==================
            # Train Generator
            # ==================
            opt_gen.zero_grad()

            # Generate fake images again
            fake_image = generator(mask)

            # Adversarial loss
            fake_pred = discriminator(mask, fake_image)
            loss_gan = criterion_gan(fake_pred, torch.ones_like(fake_pred))

            # L1 loss (reconstruction)
            loss_l1 = criterion_l1(fake_image, real_image)

            # Total generator loss
            loss_gen = loss_gan + config.lambda_l1 * loss_l1
            loss_gen.backward()
            opt_gen.step()

            # Update statistics
            epoch_g_loss += loss_gen.item()
            epoch_d_loss += loss_disc.item()
            epoch_l1_loss += loss_l1.item()

            # Update progress bar
            loop.set_postfix(
                G_loss=f"{loss_gen.item():.4f}",
                D_loss=f"{loss_disc.item():.4f}",
                L1=f"{loss_l1.item():.4f}"
            )

        # Calculate epoch averages
        avg_g_loss = epoch_g_loss / len(dataloader)
        avg_d_loss = epoch_d_loss / len(dataloader)
        avg_l1_loss = epoch_l1_loss / len(dataloader)

        print(f'\nEpoch [{epoch+1}/{config.num_epochs}] Summary:')
        print(f'  G Loss: {avg_g_loss:.4f} | D Loss: {avg_d_loss:.4f} | L1: {avg_l1_loss:.4f}')

        # Save checkpoint
        if (epoch + 1) % config.save_checkpoint_every == 0:
            checkpoint_path = os.path.join(config.checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_gen_state_dict': opt_gen.state_dict(),
                'optimizer_disc_state_dict': opt_disc.state_dict(),
                'avg_g_loss': avg_g_loss,
                'avg_d_loss': avg_d_loss,
            }, checkpoint_path)
            print(f'  ✓ Checkpoint saved: {checkpoint_path}')

        # Visualize progress
        if (epoch + 1) % config.visualize_every == 0:
            generator.eval()
            with torch.no_grad():
                mask_sample, real_sample = next(iter(dataloader))
                mask_sample = mask_sample.to(config.device)
                real_sample = real_sample.to(config.device)
                fake_sample = generator(mask_sample)

                visualize_progress(
                    mask_sample, real_sample, fake_sample,
                    epoch + 1,
                    f'progress_epoch_{epoch+1}.png'
                )
            print(f'  ✓ Progress visualization saved')
            generator.train()

    # Save final model
    final_path = os.path.join(config.checkpoint_dir, 'final_generator.pth')
    torch.save(generator.state_dict(), final_path)

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Final model saved: {final_path}\n")

    return generator


In [ ]:
def generate_new_images(generator, dataset, config):
    """Generate new synthetic fundus images with modified neovascular patterns"""

    generator.eval()
    os.makedirs(config.output_dir, exist_ok=True)

    print("\n" + "="*60)
    print(f"GENERATING {config.num_generate} SYNTHETIC IMAGES")
    print("="*60)

    with torch.no_grad():
        for i in tqdm(range(config.num_generate), desc="Generating images"):
            # Randomly select a mask
            idx = np.random.randint(0, len(dataset))
            mask, _ = dataset[idx]
            mask = mask.unsqueeze(0)  # Add batch dimension

            # Apply random transformations for variation
            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[3])  # Horizontal flip

            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[2])  # Vertical flip

            # Random rotation (0°, 90°, 180°, 270°)
            k = np.random.randint(0, 4)
            mask = torch.rot90(mask, k=k, dims=[2, 3])

            # Generate image
            mask = mask.to(config.device)
            fake_image = generator(mask)

            # Denormalize from [-1, 1] to [0, 255]
            fake_image = (fake_image.squeeze().cpu().permute(1, 2, 0).numpy() + 1) / 2
            fake_image = np.clip(fake_image * 255, 0, 255).astype(np.uint8)

            mask_np = (mask.squeeze().cpu().permute(1, 2, 0).numpy() + 1) / 2
            mask_np = np.clip(mask_np * 255, 0, 255).astype(np.uint8)

            # Save generated image and mask
            Image.fromarray(fake_image).save(
                os.path.join(config.output_dir, f'generated_fundus_{i:04d}.png')
            )
            Image.fromarray(mask_np).save(
                os.path.join(config.output_dir, f'generated_mask_{i:04d}.png')
            )

    print(f"\n✓ Generated {config.num_generate} images")
    print(f"✓ Saved to: {config.output_dir}/")
    print("="*60 + "\n")


In [ ]:
def main():
    """Main execution function"""

    print("\n" + "="*70)
    print(" "*10 + "Pix2Pix GAN for Neovascularization Fundus Generation")
    print(" "*20 + "512x512 | RGB Images | RGB Masks")
    print("="*70)
    print(f"Device: {config.device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("="*70 + "\n")

    # Step 1: Extract uploaded zip files
    print("STEP 1: Extracting Data")
    print("-" * 60)

    images_dir = 'images'
    masks_dir = 'masks'

    if os.path.exists(config.images_zip):
        with ZipFile(config.images_zip, 'r') as zip_ref:
            zip_ref.extractall(images_dir)
        print(f"✓ Extracted {config.images_zip} to {images_dir}/")
    else:
        print(f"❌ Error: {config.images_zip} not found!")
        print("   Please upload images.zip file")
        return

    if os.path.exists(config.masks_zip):
        with ZipFile(config.masks_zip, 'r') as zip_ref:
            zip_ref.extractall(masks_dir)
        print(f"✓ Extracted {config.masks_zip} to {masks_dir}/")
    else:
        print(f"❌ Error: {config.masks_zip} not found!")
        print("   Please upload masks.zip file")
        return

    # Find actual image directories (handle nested folders)
    img_subdirs = [d for d in glob.glob(os.path.join(images_dir, '**'), recursive=True)
                   if os.path.isdir(d) and len(os.listdir(d)) > 0]
    mask_subdirs = [d for d in glob.glob(os.path.join(masks_dir, '**'), recursive=True)
                    if os.path.isdir(d) and len(os.listdir(d)) > 0]

    actual_images_dir = max(img_subdirs, key=lambda x: len(x.split(os.sep))) if img_subdirs else images_dir
    actual_masks_dir = max(mask_subdirs, key=lambda x: len(x.split(os.sep))) if mask_subdirs else masks_dir

    print(f"\nData directories:")
    print(f"  Images: {actual_images_dir}")
    print(f"  Masks: {actual_masks_dir}")

    # Step 2: Create dataset
    print("\n" + "-"*60)
    print("STEP 2: Creating Dataset")
    print("-" * 60)

    dataset = RetinalDataset(
        actual_images_dir,
        actual_masks_dir,
        img_size=config.img_size
    )

    if len(dataset) == 0:
        print("❌ Error: No valid image-mask pairs found!")
        print(f"   Images in {actual_images_dir}: {len(os.listdir(actual_images_dir))}")
        print(f"   Masks in {actual_masks_dir}: {len(os.listdir(actual_masks_dir))}")
        print("\n   Troubleshooting:")
        print("   1. Ensure filenames match (extensions can differ)")
        print("   2. Check that files are valid images")
        return

    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True if config.device.type == 'cuda' else False
    )

    print(f"✓ DataLoader created")
    print(f"  Batches per epoch: {len(dataloader)}")

    # Step 3: Train the model
    print("\n" + "-"*60)
    print("STEP 3: Training Pix2Pix GAN")
    print("-" * 60)

    generator = train_pix2pix(dataloader, config)

    # Step 4: Generate new images
    print("-"*60)
    print("STEP 4: Generating Synthetic Images")
    print("-" * 60)

    generate_new_images(generator, dataset, config)

    # Final summary
    print("\n" + "="*70)
    print(" "*25 + "PIPELINE COMPLETE!")
    print("="*70)
    print(f"\n📁 Generated images: {config.output_dir}/")
    print(f"💾 Model checkpoints: {config.checkpoint_dir}/")
    print(f"📊 Progress visualizations: progress_epoch_*.png")
    print("\nNext steps:")
    print("  1. Review generated images in generated_images/")
    print("  2. Use these for data augmentation in your segmentation training")
    print("  3. Combine with original images for best results")
    print("\n" + "="*70 + "\n")

if __name__ == "__main__":
    main()


          Pix2Pix GAN for Neovascularization Fundus Generation
                    512x512 | RGB Images | RGB Masks
Device: cuda
GPU: Tesla T4
GPU Memory: 15.83 GB

STEP 1: Extracting Data
------------------------------------------------------------
✓ Extracted images.zip to images/
✓ Extracted masks.zip to masks/

Data directories:
  Images: images/
  Masks: masks/

------------------------------------------------------------
STEP 2: Creating Dataset
------------------------------------------------------------

Dataset Statistics:
✓ Found 0 valid image-mask pairs
  Image size: 512x512
  Image channels: 3 (RGB)
  Mask channels: 3 (RGB)

❌ Error: No valid image-mask pairs found!
   Images in images/: 1
   Masks in masks/: 1

   Troubleshooting:
   1. Ensure filenames match (extensions can differ)
   2. Check that files are valid images


dlfwjf

In [ ]:
"""
Pix2Pix GAN for Neovascularization Fundus Image Generation
Updated Version: 512x512, RGB Images, 3-channel masks
Complete implementation for Google Colab
"""

# ============================================================================
# ALL IMPORTS - BEGINNING
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from PIL import Image
import numpy as np
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
from zipfile import ZipFile
import glob
import random
import shutil
from pathlib import Path
import cv2

print("✓ All libraries imported successfully")

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for Pix2Pix training"""

    # Paths
    images_zip = 'images.zip'  # RGB fundus images
    masks_zip = 'masks.zip'    # RGB neovascularization masks
    output_dir = 'generated_images'
    checkpoint_dir = 'checkpoints'

    # Image parameters
    img_size = 512  # 512x512 resolution
    img_channels = 3  # RGB images
    mask_channels = 3  # RGB masks

    # Training parameters
    batch_size = 2  # Reduced for 512x512 (increase to 4 if you have enough GPU memory)
    num_epochs = 200
    lr = 0.0002
    beta1 = 0.5
    beta2 = 0.999

    # Loss weights
    lambda_l1 = 100  # L1 loss weight (structure preservation)
    lambda_perceptual = 0.1  # Perceptual loss weight

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Generation
    num_generate = 100  # Number of new synthetic images to generate

    # Save frequency
    save_checkpoint_every = 20  # Save checkpoint every N epochs
    visualize_every = 10  # Visualize results every N epochs

config = Config()

# ============================================================================
# DATASET - Updated for RGB masks
# ============================================================================

class RetinalDataset(Dataset):
    """
    Dataset for RGB fundus images and RGB neovascularization masks
    Input: RGB fundus image (3 channels)
    Condition: RGB neovascularization mask (3 channels)
    Output: RGB fundus image with neovascular vessels
    """

    def __init__(self, image_dir, mask_dir, img_size=512):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        # Get all image files (common extensions)
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        self.image_files = []
        for ext in image_extensions:
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext)))
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext.upper())))
        self.image_files = sorted(self.image_files)

        # Get all mask files
        mask_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        self.mask_files = []
        for ext in mask_extensions:
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext)))
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext.upper())))
        self.mask_files = sorted(self.mask_files)

        # Create dictionary for fast lookup
        mask_dict = {}
        for mask_path in self.mask_files:
            mask_basename = os.path.splitext(os.path.basename(mask_path))[0]
            mask_dict[mask_basename] = mask_path

        # Match images with masks by basename (extension-agnostic)
        valid_pairs = []
        unmatched_images = []

        for img_path in self.image_files:
            img_basename = os.path.splitext(os.path.basename(img_path))[0]
            if img_basename in mask_dict:
                valid_pairs.append((img_path, mask_dict[img_basename]))
            else:
                unmatched_images.append(img_basename)

        self.pairs = valid_pairs

        print(f"\n{'='*60}")
        print(f"Dataset Statistics:")
        print(f"{'='*60}")
        print(f"✓ Found {len(self.pairs)} valid image-mask pairs")
        print(f"  Image size: {img_size}x{img_size}")
        print(f"  Image channels: 3 (RGB)")
        print(f"  Mask channels: 3 (RGB)")

        if len(unmatched_images) > 0:
            print(f"\n⚠️ Warning: {len(unmatched_images)} images without matching masks:")
            for name in unmatched_images[:5]:
                print(f"     - {name}")
            if len(unmatched_images) > 5:
                print(f"     ... and {len(unmatched_images) - 5} more")

        if len(self.pairs) > 0:
            img_file = os.path.basename(self.pairs[0][0])
            mask_file = os.path.basename(self.pairs[0][1])
            print(f"\nExample pair:")
            print(f"  Image: {img_file}")
            print(f"  Mask:  {mask_file}")
        print(f"{'='*60}\n")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # Load RGB images (both fundus and mask are RGB)
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('RGB')  # RGB mask

        # Resize
        image = image.resize((self.img_size, self.img_size), Image.BICUBIC)
        mask = mask.resize((self.img_size, self.img_size), Image.BICUBIC)

        # Convert to tensors
        image = transforms.ToTensor()(image)  # [3, H, W]
        mask = transforms.ToTensor()(mask)    # [3, H, W]

        # Normalize to [-1, 1]
        image = image * 2 - 1
        mask = mask * 2 - 1

        return mask, image  # (condition, target)

# ============================================================================
# GENERATOR (U-Net) - Updated for RGB inputs
# ============================================================================

class UNetBlock(nn.Module):
    """U-Net encoder-decoder block"""

    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()

        if down:
            # Encoder
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, inplace=True)
            )
        else:
            # Decoder
            layers = [
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            if use_dropout:
                layers.append(nn.Dropout(0.5))
            self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.conv(x)

class Generator(nn.Module):
    """
    U-Net Generator for Pix2Pix
    Input: RGB neovascularization mask (3 channels)
    Output: RGB fundus image with neovascular vessels (3 channels)
    """

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        # Encoder (downsampling)
        self.down1 = nn.Conv2d(in_channels, 64, 4, 2, 1)  # 512 -> 256
        self.down2 = UNetBlock(64, 128, down=True)        # 256 -> 128
        self.down3 = UNetBlock(128, 256, down=True)       # 128 -> 64
        self.down4 = UNetBlock(256, 512, down=True)       # 64 -> 32
        self.down5 = UNetBlock(512, 512, down=True)       # 32 -> 16
        self.down6 = UNetBlock(512, 512, down=True)       # 16 -> 8
        self.down7 = UNetBlock(512, 512, down=True)       # 8 -> 4
        self.down8 = UNetBlock(512, 512, down=True)       # 4 -> 2

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, 4, 2, 1),                 # 2 -> 1
            nn.ReLU(inplace=True)
        )

        # Decoder (upsampling)
        self.up1 = UNetBlock(512, 512, down=False, use_dropout=True)      # 1 -> 2
        self.up2 = UNetBlock(1024, 512, down=False, use_dropout=True)     # 2 -> 4
        self.up3 = UNetBlock(1024, 512, down=False, use_dropout=True)     # 4 -> 8
        self.up4 = UNetBlock(1024, 512, down=False)                       # 8 -> 16
        self.up5 = UNetBlock(1024, 512, down=False)                       # 16 -> 32
        self.up6 = UNetBlock(1024, 256, down=False)                       # 32 -> 64
        self.up7 = UNetBlock(512, 128, down=False)                        # 64 -> 128
        self.up8 = UNetBlock(256, 64, down=False)                         # 128 -> 256

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),               # 256 -> 512
            nn.Tanh()
        )

    def forward(self, x):
        # Encoder with skip connections
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        # Bottleneck
        bottleneck = self.bottleneck(d8)

        # Decoder with skip connections
        u1 = self.up1(bottleneck)
        u2 = self.up2(torch.cat([u1, d8], dim=1))
        u3 = self.up3(torch.cat([u2, d7], dim=1))
        u4 = self.up4(torch.cat([u3, d6], dim=1))
        u5 = self.up5(torch.cat([u4, d5], dim=1))
        u6 = self.up6(torch.cat([u5, d4], dim=1))
        u7 = self.up7(torch.cat([u6, d3], dim=1))
        u8 = self.up8(torch.cat([u7, d2], dim=1))

        return self.final(torch.cat([u8, d1], dim=1))

# ============================================================================
# DISCRIMINATOR (PatchGAN) - Updated for RGB inputs
# ============================================================================

class Discriminator(nn.Module):
    """
    PatchGAN Discriminator
    Input: Concatenated RGB mask (3) + RGB image (3) = 6 channels
    """

    def __init__(self, in_channels=6):  # mask (3) + image (3)
        super().__init__()

        self.model = nn.Sequential(
            # Input: 512x512
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            # 256x256
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 128x128
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # 64x64
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32
            nn.Conv2d(512, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # 31x31
            nn.Conv2d(512, 1, 4, 1, 1)
            # Output: 30x30 patch predictions
        )

    def forward(self, mask, image):
        x = torch.cat([mask, image], dim=1)  # Concatenate along channel dimension
        return self.model(x)

# ============================================================================
# TRAINING
# ============================================================================

def save_sample_images(mask, real, fake, epoch, batch_idx, save_dir='samples'):
    """Save sample images during training"""
    os.makedirs(save_dir, exist_ok=True)

    # Denormalize from [-1, 1] to [0, 1]
    mask = (mask + 1) / 2
    real = (real + 1) / 2
    fake = (fake + 1) / 2

    # Create grid
    comparison = torch.cat([mask[:4], real[:4], fake[:4]], dim=0)
    grid = make_grid(comparison, nrow=4, padding=2, normalize=False)

    # Save
    save_path = os.path.join(save_dir, f'epoch_{epoch}_batch_{batch_idx}.png')
    save_image(grid, save_path)

def visualize_progress(mask, real_image, fake_image, epoch, save_path='progress.png'):
    """Visualize training progress"""

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    num_samples = min(4, mask.size(0))

    for i in range(num_samples):
        # Denormalize
        mask_np = (mask[i].cpu().permute(1, 2, 0).numpy() + 1) / 2
        real_np = (real_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2
        fake_np = (fake_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2

        # Clip to valid range
        mask_np = np.clip(mask_np, 0, 1)
        real_np = np.clip(real_np, 0, 1)
        fake_np = np.clip(fake_np, 0, 1)

        # Plot
        axes[0, i].imshow(mask_np)
        axes[0, i].set_title(f'Neovascular Mask {i+1}', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(real_np)
        axes[1, i].set_title(f'Real Fundus {i+1}', fontsize=10)
        axes[1, i].axis('off')

        axes[2, i].imshow(fake_np)
        axes[2, i].set_title(f'Generated Fundus {i+1}', fontsize=10)
        axes[2, i].axis('off')

    plt.suptitle(f'Training Progress - Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def train_pix2pix(dataloader, config):
    """Train Pix2Pix GAN"""

    print("\n" + "="*60)
    print("INITIALIZING TRAINING")
    print("="*60)

    # Initialize models
    generator = Generator(in_channels=3, out_channels=3).to(config.device)
    discriminator = Discriminator(in_channels=6).to(config.device)

    # Count parameters
    gen_params = sum(p.numel() for p in generator.parameters() if p.requires_grad)
    disc_params = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
    print(f"\nModel Architecture:")
    print(f"  Generator parameters: {gen_params:,}")
    print(f"  Discriminator parameters: {disc_params:,}")

    # Optimizers
    opt_gen = optim.Adam(generator.parameters(), lr=config.lr, betas=(config.beta1, config.beta2))
    opt_disc = optim.Adam(discriminator.parameters(), lr=config.lr, betas=(config.beta1, config.beta2))

    # Loss functions
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_l1 = nn.L1Loss()

    # Create directories
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    os.makedirs('samples', exist_ok=True)

    print(f"\nTraining Configuration:")
    print(f"  Device: {config.device}")
    print(f"  Epochs: {config.num_epochs}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Learning rate: {config.lr}")
    print(f"  Image size: {config.img_size}x{config.img_size}")
    print(f"  L1 weight: {config.lambda_l1}")
    print("="*60 + "\n")

    # Training loop
    for epoch in range(config.num_epochs):
        generator.train()
        discriminator.train()

        loop = tqdm(dataloader, desc=f'Epoch {epoch+1}/{config.num_epochs}')

        epoch_g_loss = 0
        epoch_d_loss = 0
        epoch_l1_loss = 0

        for batch_idx, (mask, real_image) in enumerate(loop):
            mask = mask.to(config.device)
            real_image = real_image.to(config.device)
            batch_size = mask.size(0)

            # ==================
            # Train Discriminator
            # ==================
            opt_disc.zero_grad()

            # Generate fake images
            fake_image = generator(mask)

            # Real images
            real_pred = discriminator(mask, real_image)
            real_labels = torch.ones_like(real_pred)
            loss_real = criterion_gan(real_pred, real_labels)

            # Fake images
            fake_pred = discriminator(mask, fake_image.detach())
            fake_labels = torch.zeros_like(fake_pred)
            loss_fake = criterion_gan(fake_pred, fake_labels)

            # Total discriminator loss
            loss_disc = (loss_real + loss_fake) * 0.5
            loss_disc.backward()
            opt_disc.step()

            # ==================
            # Train Generator
            # ==================
            opt_gen.zero_grad()

            # Generate fake images again
            fake_image = generator(mask)

            # Adversarial loss
            fake_pred = discriminator(mask, fake_image)
            loss_gan = criterion_gan(fake_pred, torch.ones_like(fake_pred))

            # L1 loss (reconstruction)
            loss_l1 = criterion_l1(fake_image, real_image)

            # Total generator loss
            loss_gen = loss_gan + config.lambda_l1 * loss_l1
            loss_gen.backward()
            opt_gen.step()

            # Update statistics
            epoch_g_loss += loss_gen.item()
            epoch_d_loss += loss_disc.item()
            epoch_l1_loss += loss_l1.item()

            # Update progress bar
            loop.set_postfix(
                G_loss=f"{loss_gen.item():.4f}",
                D_loss=f"{loss_disc.item():.4f}",
                L1=f"{loss_l1.item():.4f}"
            )

        # Calculate epoch averages
        avg_g_loss = epoch_g_loss / len(dataloader)
        avg_d_loss = epoch_d_loss / len(dataloader)
        avg_l1_loss = epoch_l1_loss / len(dataloader)

        print(f'\nEpoch [{epoch+1}/{config.num_epochs}] Summary:')
        print(f'  G Loss: {avg_g_loss:.4f} | D Loss: {avg_d_loss:.4f} | L1: {avg_l1_loss:.4f}')

        # Save checkpoint
        if (epoch + 1) % config.save_checkpoint_every == 0:
            checkpoint_path = os.path.join(config.checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_gen_state_dict': opt_gen.state_dict(),
                'optimizer_disc_state_dict': opt_disc.state_dict(),
                'avg_g_loss': avg_g_loss,
                'avg_d_loss': avg_d_loss,
            }, checkpoint_path)
            print(f'  ✓ Checkpoint saved: {checkpoint_path}')

        # Visualize progress
        if (epoch + 1) % config.visualize_every == 0:
            generator.eval()
            with torch.no_grad():
                mask_sample, real_sample = next(iter(dataloader))
                mask_sample = mask_sample.to(config.device)
                real_sample = real_sample.to(config.device)
                fake_sample = generator(mask_sample)

                visualize_progress(
                    mask_sample, real_sample, fake_sample,
                    epoch + 1,
                    f'progress_epoch_{epoch+1}.png'
                )
            print(f'  ✓ Progress visualization saved')
            generator.train()

    # Save final model
    final_path = os.path.join(config.checkpoint_dir, 'final_generator.pth')
    torch.save(generator.state_dict(), final_path)

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Final model saved: {final_path}\n")

    return generator

# ============================================================================
# GENERATION
# ============================================================================

def generate_new_images(generator, dataset, config):
    """Generate new synthetic fundus images with modified neovascular patterns"""

    generator.eval()
    os.makedirs(config.output_dir, exist_ok=True)

    print("\n" + "="*60)
    print(f"GENERATING {config.num_generate} SYNTHETIC IMAGES")
    print("="*60)

    with torch.no_grad():
        for i in tqdm(range(config.num_generate), desc="Generating images"):
            # Randomly select a mask
            idx = np.random.randint(0, len(dataset))
            mask, _ = dataset[idx]
            mask = mask.unsqueeze(0)  # Add batch dimension

            # Apply random transformations for variation
            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[3])  # Horizontal flip

            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[2])  # Vertical flip

            # Random rotation (0°, 90°, 180°, 270°)
            k = np.random.randint(0, 4)
            mask = torch.rot90(mask, k=k, dims=[2, 3])

            # Generate image
            mask = mask.to(config.device)
            fake_image = generator(mask)

            # Denormalize from [-1, 1] to [0, 255]
            fake_image = (fake_image.squeeze().cpu().permute(1, 2, 0).numpy() + 1) / 2
            fake_image = np.clip(fake_image * 255, 0, 255).astype(np.uint8)

            mask_np = (mask.squeeze().cpu().permute(1, 2, 0).numpy() + 1) / 2
            mask_np = np.clip(mask_np * 255, 0, 255).astype(np.uint8)

            # Save generated image and mask
            Image.fromarray(fake_image).save(
                os.path.join(config.output_dir, f'generated_fundus_{i:04d}.png')
            )
            Image.fromarray(mask_np).save(
                os.path.join(config.output_dir, f'generated_mask_{i:04d}.png')
            )

    print(f"\n✓ Generated {config.num_generate} images")
    print(f"✓ Saved to: {config.output_dir}/")
    print("="*60 + "\n")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""

    print("\n" + "="*70)
    print(" "*10 + "Pix2Pix GAN for Neovascularization Fundus Generation")
    print(" "*20 + "512x512 | RGB Images | RGB Masks")
    print("="*70)
    print(f"Device: {config.device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("="*70 + "\n")

    # Step 1: Extract uploaded zip files
    print("STEP 1: Extracting Data")
    print("-" * 60)

    images_dir = 'images'
    masks_dir = 'masks'

    if os.path.exists(config.images_zip):
        with ZipFile(config.images_zip, 'r') as zip_ref:
            zip_ref.extractall(images_dir)
        print(f"✓ Extracted {config.images_zip} to {images_dir}/")
    else:
        print(f"❌ Error: {config.images_zip} not found!")
        print("   Please upload images.zip file")
        return

    if os.path.exists(config.masks_zip):
        with ZipFile(config.masks_zip, 'r') as zip_ref:
            zip_ref.extractall(masks_dir)
        print(f"✓ Extracted {config.masks_zip} to {masks_dir}/")
    else:
        print(f"❌ Error: {config.masks_zip} not found!")
        print("   Please upload masks.zip file")
        return

    # Find actual image directories (handle nested folders)
    # Check if there's a nested 'images' folder inside the extracted 'images' directory
    nested_images = os.path.join(images_dir, 'images')
    if os.path.exists(nested_images) and os.path.isdir(nested_images):
        actual_images_dir = nested_images
    else:
        # Fall back to finding any subdirectory with files
        img_subdirs = [d for d in glob.glob(os.path.join(images_dir, '**'), recursive=True)
                       if os.path.isdir(d) and len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]) > 0]
        actual_images_dir = max(img_subdirs, key=lambda x: len(x.split(os.sep))) if img_subdirs else images_dir

    # Check if there's a nested 'masks' folder inside the extracted 'masks' directory
    nested_masks = os.path.join(masks_dir, 'masks')
    if os.path.exists(nested_masks) and os.path.isdir(nested_masks):
        actual_masks_dir = nested_masks
    else:
        # Fall back to finding any subdirectory with files
        mask_subdirs = [d for d in glob.glob(os.path.join(masks_dir, '**'), recursive=True)
                        if os.path.isdir(d) and len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]) > 0]
        actual_masks_dir = max(mask_subdirs, key=lambda x: len(x.split(os.sep))) if mask_subdirs else masks_dir

    print(f"\nData directories:")
    print(f"  Images: {actual_images_dir}")
    print(f"  Masks: {actual_masks_dir}")

    # Step 2: Create dataset
    print("\n" + "-"*60)
    print("STEP 2: Creating Dataset")
    print("-" * 60)

    dataset = RetinalDataset(
        actual_images_dir,
        actual_masks_dir,
        img_size=config.img_size
    )

    if len(dataset) == 0:
        print("❌ Error: No valid image-mask pairs found!")
        print(f"   Images in {actual_images_dir}: {len(os.listdir(actual_images_dir))}")
        print(f"   Masks in {actual_masks_dir}: {len(os.listdir(actual_masks_dir))}")
        print("\n   Troubleshooting:")
        print("   1. Ensure filenames match (extensions can differ)")
        print("   2. Check that files are valid images")
        return

    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True if config.device.type == 'cuda' else False
    )

    print(f"✓ DataLoader created")
    print(f"  Batches per epoch: {len(dataloader)}")

    # Step 3: Train the model
    print("\n" + "-"*60)
    print("STEP 3: Training Pix2Pix GAN")
    print("-" * 60)

    generator = train_pix2pix(dataloader, config)

    # Step 4: Generate new images
    print("-"*60)
    print("STEP 4: Generating Synthetic Images")
    print("-" * 60)

    generate_new_images(generator, dataset, config)

    # Final summary
    print("\n" + "="*70)
    print(" "*25 + "PIPELINE COMPLETE!")
    print("="*70)
    print(f"\n📁 Generated images: {config.output_dir}/")
    print(f"💾 Model checkpoints: {config.checkpoint_dir}/")
    print(f"📊 Progress visualizations: progress_epoch_*.png")
    print("\nNext steps:")
    print("  1. Review generated images in generated_images/")
    print("  2. Use these for data augmentation in your segmentation training")
    print("  3. Combine with original images for best results")
    print("\n" + "="*70 + "\n")

if __name__ == "__main__":
    main()

✓ All libraries imported successfully

          Pix2Pix GAN for Neovascularization Fundus Generation
                    512x512 | RGB Images | RGB Masks
Device: cuda
GPU: Tesla T4
GPU Memory: 15.83 GB

STEP 1: Extracting Data
------------------------------------------------------------
✓ Extracted images.zip to images/
✓ Extracted masks.zip to masks/

Data directories:
  Images: images/images
  Masks: masks/masks

------------------------------------------------------------
STEP 2: Creating Dataset
------------------------------------------------------------

Dataset Statistics:
✓ Found 120 valid image-mask pairs
  Image size: 512x512
  Image channels: 3 (RGB)
  Mask channels: 3 (RGB)

Example pair:
  Image: 06051963_.jpg
  Mask:  06051963_.png

✓ DataLoader created
  Batches per epoch: 60

------------------------------------------------------------
STEP 3: Training Pix2Pix GAN
------------------------------------------------------------

INITIALIZING TRAINING

Model Architecture:
 

Epoch 1/200: 100%|██████████| 60/60 [00:26<00:00,  2.30it/s, D_loss=0.0235, G_loss=25.2176, L1=0.2137]



Epoch [1/200] Summary:
  G Loss: 35.1787 | D Loss: 0.2233 | L1: 0.3261


Epoch 2/200: 100%|██████████| 60/60 [00:25<00:00,  2.38it/s, D_loss=0.0084, G_loss=27.1935, L1=0.2242]



Epoch [2/200] Summary:
  G Loss: 29.5188 | D Loss: 0.0163 | L1: 0.2512


...


Epoch [199/200] Summary:
  G Loss: 9.8274 | D Loss: 0.1501 | L1: 0.0570


Epoch 200/200: 100%|██████████| 60/60 [00:26<00:00,  2.30it/s, D_loss=0.0147, G_loss=10.3714, L1=0.0561]



Epoch [200/200] Summary:
  G Loss: 10.2018 | D Loss: 0.1090 | L1: 0.0615
  ✓ Checkpoint saved: checkpoints/checkpoint_epoch_200.pth
  ✓ Progress visualization saved

TRAINING COMPLETE!
Final model saved: checkpoints/final_generator.pth

------------------------------------------------------------
STEP 4: Generating Synthetic Images
------------------------------------------------------------

GENERATING 100 SYNTHETIC IMAGES


Generating images: 100%|██████████| 100/100 [00:31<00:00,  3.15it/s]


✓ Generated 100 images
✓ Saved to: generated_images/


                         PIPELINE COMPLETE!

📁 Generated images: generated_images/
💾 Model checkpoints: checkpoints/
📊 Progress visualizations: progress_epoch_*.png

Next steps:
  1. Review generated images in generated_images/
  2. Use these for data augmentation in your segmentation training
  3. Combine with original images for best results




**GREY SCALE MASK IMPLEMENTATION**

In [ ]:
"""
IMPROVED Pix2Pix GAN for Neovascularization Fundus Image Generation
Version 2.0: Grayscale masks + Edge-aware + Vessel-focused losses
512x512 Resolution
"""

# ============================================================================
# ALL IMPORTS
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
from torchvision.models import vgg19
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from zipfile import ZipFile
import glob
import cv2

print("✓ All libraries imported successfully")

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Enhanced configuration"""

    # Paths
    images_zip = 'images.zip'
    masks_zip = 'masks.zip'
    output_dir = 'generated_images'
    checkpoint_dir = 'checkpoints'

    # Image parameters
    img_size = 512
    img_channels = 3  # RGB fundus images
    mask_channels = 1  # GRAYSCALE masks (this is the key change!)

    # Training parameters
    batch_size = 2
    num_epochs = 200
    lr_gen = 0.0002
    lr_disc = 0.0002
    beta1 = 0.5
    beta2 = 0.999

    # Loss weights (TUNED for vessel preservation)
    lambda_l1 = 100          # L1 reconstruction
    lambda_perceptual = 10   # Perceptual loss (increased!)
    lambda_edge = 50         # Edge-aware loss (NEW!)
    lambda_focal = 20        # Focal loss for vessels (NEW!)

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Generation
    num_generate = 100

    # Save frequency
    save_checkpoint_every = 20
    visualize_every = 10

config = Config()

# ============================================================================
# PERCEPTUAL LOSS (VGG-based)
# ============================================================================

class VGGPerceptualLoss(nn.Module):
    """Perceptual loss using VGG19 features"""

    def __init__(self):
        super().__init__()
        vgg = vgg19(pretrained=True).features
        self.slice1 = nn.Sequential(*list(vgg[:4])).eval()
        self.slice2 = nn.Sequential(*list(vgg[4:9])).eval()
        self.slice3 = nn.Sequential(*list(vgg[9:18])).eval()

        for param in self.parameters():
            param.requires_grad = False

    def forward(self, fake, real):
        # Normalize for VGG
        fake = (fake + 1) / 2  # [-1,1] -> [0,1]
        real = (real + 1) / 2

        # VGG expects ImageNet normalization
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(fake.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(fake.device)

        fake = (fake - mean) / std
        real = (real - mean) / std

        # Multi-scale features
        fake_f1 = self.slice1(fake)
        real_f1 = self.slice1(real)

        fake_f2 = self.slice2(fake_f1)
        real_f2 = self.slice2(real_f1)

        fake_f3 = self.slice3(fake_f2)
        real_f3 = self.slice3(real_f2)

        # Combine losses from multiple layers
        loss = (F.l1_loss(fake_f1, real_f1) +
                F.l1_loss(fake_f2, real_f2) +
                F.l1_loss(fake_f3, real_f3))

        return loss

# ============================================================================
# EDGE-AWARE LOSS
# ============================================================================

class EdgeAwareLoss(nn.Module):
    """Preserve vessel edges/boundaries"""

    def __init__(self):
        super().__init__()
        # Sobel edge detection kernels
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)

        self.sobel_x = sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.sobel_y = sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1)

    def forward(self, fake, real):
        self.sobel_x = self.sobel_x.to(fake.device)
        self.sobel_y = self.sobel_y.to(real.device)

        # Compute edges
        fake_edge_x = F.conv2d(fake, self.sobel_x, padding=1, groups=3)
        fake_edge_y = F.conv2d(fake, self.sobel_y, padding=1, groups=3)
        fake_edge = torch.sqrt(fake_edge_x**2 + fake_edge_y**2 + 1e-6)

        real_edge_x = F.conv2d(real, self.sobel_x, padding=1, groups=3)
        real_edge_y = F.conv2d(real, self.sobel_y, padding=1, groups=3)
        real_edge = torch.sqrt(real_edge_x**2 + real_edge_y**2 + 1e-6)

        return F.l1_loss(fake_edge, real_edge)

# ============================================================================
# FOCAL LOSS (Focus on vessel regions)
# ============================================================================

class FocalL1Loss(nn.Module):
    """Focal loss to emphasize vessel regions"""

    def __init__(self, alpha=2.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, fake, real, mask):
        """
        Args:
            fake: Generated image
            real: Real image
            mask: Vessel mask (1 channel) - vessels = 1, background = 0
        """
        # Expand mask to match image channels
        mask_3ch = mask.repeat(1, 3, 1, 1)  # [B, 1, H, W] -> [B, 3, H, W]

        # L1 difference
        diff = torch.abs(fake - real)

        # Weight by mask - higher weight for vessel regions
        # Background: weight = 1.0
        # Vessels: weight = (1 + alpha) = 3.0
        weights = 1.0 + self.alpha * mask_3ch

        weighted_diff = diff * weights

        return weighted_diff.mean()

# ============================================================================
# DATASET - GRAYSCALE MASKS
# ============================================================================

class RetinalDataset(Dataset):
    """
    Dataset with GRAYSCALE neovascularization masks
    Input: Grayscale mask (1 channel) - vessels = white, background = black
    Output: RGB fundus image (3 channels)
    """

    def __init__(self, image_dir, mask_dir, img_size=512):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        # Get image files
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        self.image_files = []
        for ext in image_extensions:
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext)))
            self.image_files.extend(glob.glob(os.path.join(image_dir, ext.upper())))
        self.image_files = sorted(self.image_files)

        # Get mask files
        self.mask_files = []
        for ext in image_extensions:
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext)))
            self.mask_files.extend(glob.glob(os.path.join(mask_dir, ext.upper())))
        self.mask_files = sorted(self.mask_files)

        # Match by basename
        mask_dict = {}
        for mask_path in self.mask_files:
            mask_basename = os.path.splitext(os.path.basename(mask_path))[0]
            mask_dict[mask_basename] = mask_path

        valid_pairs = []
        for img_path in self.image_files:
            img_basename = os.path.splitext(os.path.basename(img_path))[0]
            if img_basename in mask_dict:
                valid_pairs.append((img_path, mask_dict[img_basename]))

        self.pairs = valid_pairs

        print(f"\n{'='*60}")
        print(f"Dataset Statistics:")
        print(f"{'='*60}")
        print(f"✓ Found {len(self.pairs)} valid image-mask pairs")
        print(f"  Image size: {img_size}x{img_size}")
        print(f"  Image channels: 3 (RGB fundus)")
        print(f"  Mask channels: 1 (GRAYSCALE vessels)")

        if len(self.pairs) > 0:
            img_file = os.path.basename(self.pairs[0][0])
            mask_file = os.path.basename(self.pairs[0][1])
            print(f"\nExample pair:")
            print(f"  Image: {img_file}")
            print(f"  Mask:  {mask_file}")
        print(f"{'='*60}\n")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # Load RGB fundus image
        image = Image.open(img_path).convert('RGB')

        # Load mask as GRAYSCALE
        mask = Image.open(mask_path).convert('L')  # Grayscale!

        # Enhance mask contrast (ensure vessels are bright)
        mask_np = np.array(mask)
        # Binary threshold to ensure clear vessels
        _, mask_np = cv2.threshold(mask_np, 127, 255, cv2.THRESH_BINARY)
        mask = Image.fromarray(mask_np)

        # Resize
        image = image.resize((self.img_size, self.img_size), Image.BICUBIC)
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        # Convert to tensors
        image = transforms.ToTensor()(image)  # [3, H, W]
        mask = transforms.ToTensor()(mask)    # [1, H, W]

        # Normalize to [-1, 1]
        image = image * 2 - 1
        mask = mask * 2 - 1

        return mask, image

# ============================================================================
# GENERATOR - UPDATED FOR 1-CHANNEL INPUT
# ============================================================================

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()

        if down:
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, inplace=True)
            )
        else:
            layers = [
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            if use_dropout:
                layers.append(nn.Dropout(0.5))
            self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.conv(x)

class Generator(nn.Module):
    """
    U-Net Generator
    Input: Grayscale vessel mask (1 channel)
    Output: RGB fundus image (3 channels)
    """

    def __init__(self, in_channels=1, out_channels=3):
        super().__init__()

        # Encoder
        self.down1 = nn.Conv2d(in_channels, 64, 4, 2, 1)  # 512 -> 256
        self.down2 = UNetBlock(64, 128, down=True)        # 256 -> 128
        self.down3 = UNetBlock(128, 256, down=True)       # 128 -> 64
        self.down4 = UNetBlock(256, 512, down=True)       # 64 -> 32
        self.down5 = UNetBlock(512, 512, down=True)       # 32 -> 16
        self.down6 = UNetBlock(512, 512, down=True)       # 16 -> 8
        self.down7 = UNetBlock(512, 512, down=True)       # 8 -> 4
        self.down8 = UNetBlock(512, 512, down=True)       # 4 -> 2

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, 4, 2, 1),
            nn.ReLU(inplace=True)
        )

        # Decoder
        self.up1 = UNetBlock(512, 512, down=False, use_dropout=True)
        self.up2 = UNetBlock(1024, 512, down=False, use_dropout=True)
        self.up3 = UNetBlock(1024, 512, down=False, use_dropout=True)
        self.up4 = UNetBlock(1024, 512, down=False)
        self.up5 = UNetBlock(1024, 512, down=False)
        self.up6 = UNetBlock(1024, 256, down=False)
        self.up7 = UNetBlock(512, 128, down=False)
        self.up8 = UNetBlock(256, 64, down=False)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        bottleneck = self.bottleneck(d8)

        u1 = self.up1(bottleneck)
        u2 = self.up2(torch.cat([u1, d8], dim=1))
        u3 = self.up3(torch.cat([u2, d7], dim=1))
        u4 = self.up4(torch.cat([u3, d6], dim=1))
        u5 = self.up5(torch.cat([u4, d5], dim=1))
        u6 = self.up6(torch.cat([u5, d4], dim=1))
        u7 = self.up7(torch.cat([u6, d3], dim=1))
        u8 = self.up8(torch.cat([u7, d2], dim=1))

        return self.final(torch.cat([u8, d1], dim=1))

# ============================================================================
# DISCRIMINATOR - UPDATED FOR 1+3=4 CHANNELS
# ============================================================================

class Discriminator(nn.Module):
    """
    PatchGAN Discriminator
    Input: mask (1) + image (3) = 4 channels
    """

    def __init__(self, in_channels=4):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 1)
        )

    def forward(self, mask, image):
        x = torch.cat([mask, image], dim=1)
        return self.model(x)

# ============================================================================
# TRAINING WITH IMPROVED LOSSES
# ============================================================================

def visualize_progress(mask, real_image, fake_image, epoch, save_path='progress.png'):
    """Visualize training progress"""

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    num_samples = min(4, mask.size(0))

    for i in range(num_samples):
        # Denormalize
        mask_np = (mask[i].cpu().numpy().squeeze() + 1) / 2
        real_np = (real_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2
        fake_np = (fake_image[i].cpu().permute(1, 2, 0).numpy() + 1) / 2

        mask_np = np.clip(mask_np, 0, 1)
        real_np = np.clip(real_np, 0, 1)
        fake_np = np.clip(fake_np, 0, 1)

        axes[0, i].imshow(mask_np, cmap='gray')
        axes[0, i].set_title(f'Vessel Mask {i+1}', fontsize=10)
        axes[0, i].axis('off')

        axes[1, i].imshow(real_np)
        axes[1, i].set_title(f'Real Fundus {i+1}', fontsize=10)
        axes[1, i].axis('off')

        axes[2, i].imshow(fake_np)
        axes[2, i].set_title(f'Generated Fundus {i+1}', fontsize=10)
        axes[2, i].axis('off')

    plt.suptitle(f'Training Progress - Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def train_pix2pix(dataloader, config):
    """Train with improved losses"""

    print("\n" + "="*60)
    print("INITIALIZING IMPROVED TRAINING")
    print("="*60)

    # Models
    generator = Generator(in_channels=1, out_channels=3).to(config.device)
    discriminator = Discriminator(in_channels=4).to(config.device)

    # Optimizers
    opt_gen = optim.Adam(generator.parameters(), lr=config.lr_gen, betas=(config.beta1, config.beta2))
    opt_disc = optim.Adam(discriminator.parameters(), lr=config.lr_disc, betas=(config.beta1, config.beta2))

    # Loss functions
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_l1 = nn.L1Loss()
    criterion_perceptual = VGGPerceptualLoss().to(config.device)
    criterion_edge = EdgeAwareLoss().to(config.device)
    criterion_focal = FocalL1Loss(alpha=2.0).to(config.device)

    os.makedirs(config.checkpoint_dir, exist_ok=True)

    print(f"\nLoss Configuration:")
    print(f"  L1 weight: {config.lambda_l1}")
    print(f"  Perceptual weight: {config.lambda_perceptual}")
    print(f"  Edge weight: {config.lambda_edge}")
    print(f"  Focal weight: {config.lambda_focal}")
    print("="*60 + "\n")

    # Training loop
    for epoch in range(config.num_epochs):
        generator.train()
        discriminator.train()

        loop = tqdm(dataloader, desc=f'Epoch {epoch+1}/{config.num_epochs}')

        epoch_losses = {
            'g_total': 0, 'd_total': 0, 'l1': 0,
            'perceptual': 0, 'edge': 0, 'focal': 0
        }

        for batch_idx, (mask, real_image) in enumerate(loop):
            mask = mask.to(config.device)
            real_image = real_image.to(config.device)

            # Train Discriminator
            opt_disc.zero_grad()
            fake_image = generator(mask)

            real_pred = discriminator(mask, real_image)
            loss_real = criterion_gan(real_pred, torch.ones_like(real_pred))

            fake_pred = discriminator(mask, fake_image.detach())
            loss_fake = criterion_gan(fake_pred, torch.zeros_like(fake_pred))

            loss_disc = (loss_real + loss_fake) * 0.5
            loss_disc.backward()
            opt_disc.step()

            # Train Generator with MULTIPLE LOSSES
            opt_gen.zero_grad()
            fake_image = generator(mask)

            # 1. Adversarial loss
            fake_pred = discriminator(mask, fake_image)
            loss_gan = criterion_gan(fake_pred, torch.ones_like(fake_pred))

            # 2. L1 loss
            loss_l1 = criterion_l1(fake_image, real_image)

            # 3. Perceptual loss (VGG features)
            loss_perceptual = criterion_perceptual(fake_image, real_image)

            # 4. Edge-aware loss
            loss_edge = criterion_edge(fake_image, real_image)

            # 5. Focal loss (emphasize vessel regions)
            # Convert mask from [-1,1] to [0,1] for focal loss
            mask_01 = (mask + 1) / 2
            loss_focal = criterion_focal(fake_image, real_image, mask_01)

            # Total generator loss
            loss_gen = (loss_gan +
                       config.lambda_l1 * loss_l1 +
                       config.lambda_perceptual * loss_perceptual +
                       config.lambda_edge * loss_edge +
                       config.lambda_focal * loss_focal)

            loss_gen.backward()
            opt_gen.step()

            # Update statistics
            epoch_losses['g_total'] += loss_gen.item()
            epoch_losses['d_total'] += loss_disc.item()
            epoch_losses['l1'] += loss_l1.item()
            epoch_losses['perceptual'] += loss_perceptual.item()
            epoch_losses['edge'] += loss_edge.item()
            epoch_losses['focal'] += loss_focal.item()

            loop.set_postfix(
                G=f"{loss_gen.item():.3f}",
                D=f"{loss_disc.item():.3f}",
                L1=f"{loss_l1.item():.3f}",
                Edge=f"{loss_edge.item():.3f}"
            )

        # Epoch summary
        n_batches = len(dataloader)
        print(f'\nEpoch [{epoch+1}/{config.num_epochs}] Summary:')
        print(f'  G: {epoch_losses["g_total"]/n_batches:.4f} | '
              f'D: {epoch_losses["d_total"]/n_batches:.4f} | '
              f'L1: {epoch_losses["l1"]/n_batches:.4f}')
        print(f'  Perceptual: {epoch_losses["perceptual"]/n_batches:.4f} | '
              f'Edge: {epoch_losses["edge"]/n_batches:.4f} | '
              f'Focal: {epoch_losses["focal"]/n_batches:.4f}')

        # Save checkpoint
        if (epoch + 1) % config.save_checkpoint_every == 0:
            checkpoint_path = os.path.join(config.checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
            }, checkpoint_path)
            print(f'  ✓ Checkpoint saved')

        # Visualize
        if (epoch + 1) % config.visualize_every == 0:
            generator.eval()
            with torch.no_grad():
                mask_sample, real_sample = next(iter(dataloader))
                mask_sample = mask_sample.to(config.device)
                real_sample = real_sample.to(config.device)
                fake_sample = generator(mask_sample)
                visualize_progress(mask_sample, real_sample, fake_sample,
                                 epoch + 1, f'progress_epoch_{epoch+1}.png')
            print(f'  ✓ Progress saved')
            generator.train()

    # Save final
    torch.save(generator.state_dict(), os.path.join(config.checkpoint_dir, 'final_generator.pth'))

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60 + "\n")

    return generator

# ============================================================================
# GENERATION
# ============================================================================

def generate_new_images(generator, dataset, config):
    """Generate synthetic images"""

    generator.eval()
    os.makedirs(config.output_dir, exist_ok=True)

    print(f"\nGenerating {config.num_generate} synthetic images...")

    with torch.no_grad():
        for i in tqdm(range(config.num_generate)):
            idx = np.random.randint(0, len(dataset))
            mask, _ = dataset[idx]
            mask = mask.unsqueeze(0)

            # Random augmentations
            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[3])
            if np.random.rand() > 0.5:
                mask = torch.flip(mask, dims=[2])
            k = np.random.randint(0, 4)
            mask = torch.rot90(mask, k=k, dims=[2, 3])

            mask = mask.to(config.device)
            fake_image = generator(mask)

            # Save
            fake_image = (fake_image.squeeze().cpu().permute(1, 2, 0).numpy() + 1) / 2
            fake_image = np.clip(fake_image * 255, 0, 255).astype(np.uint8)

            mask_np = (mask.squeeze().cpu().numpy() + 1) / 2
            mask_np = np.clip(mask_np * 255, 0, 255).astype(np.uint8)

            Image.fromarray(fake_image).save(os.path.join(config.output_dir, f'generated_fundus_{i:04d}.png'))
            Image.fromarray(mask_np).save(os.path.join(config.output_dir, f'generated_mask_{i:04d}.png'))

    print(f"✓ Generated images saved to {config.output_dir}/\n")

# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main execution"""

    print("\n" + "="*70)
    print(" "*5 + "IMPROVED Pix2Pix GAN for Neovascularization Generation")
    print(" "*15 + "Grayscale Masks + Edge-Aware + Vessel-Focused")
    print("="*70)
    print(f"Device: {config.device}\n")

    # Extract data
    images_dir = 'images'
    masks_dir = 'masks'

    if os.path.exists(config.images_zip):
        with ZipFile(config.images_zip, 'r') as zip_ref:
            zip_ref.extractall(images_dir)
        print(f"✓ Extracted images")

    if os.path.exists(config.masks_zip):
        with ZipFile(config.masks_zip, 'r') as zip_ref:
            zip_ref.extractall(masks_dir)
        print(f"✓ Extracted masks")

    # Find directories (handle nested)
    nested_images = os.path.join(images_dir, 'images')
    actual_images_dir = nested_images if os.path.exists(nested_images) else images_dir

    nested_masks = os.path.join(masks_dir, 'masks')
    actual_masks_dir = nested_masks if os.path.exists(nested_masks) else masks_dir

    # Create dataset
    dataset = RetinalDataset(actual_images_dir, actual_masks_dir, img_size=config.img_size)

    if len(dataset) == 0:
        print("❌ No valid pairs found!")
        return

    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=2,
                           pin_memory=True if config.device.type == 'cuda' else False)

    # Train
    print("\nStarting training with improved losses...\n")
    generator = train_pix2pix(dataloader, config)

    # Generate
    generate_new_images(generator, dataset, config)

    print("="*70)
    print("PIPELINE COMPLETE!")
    print(f"Generated images: {config.output_dir}/")
    print(f"Checkpoints: {config.checkpoint_dir}/")
    print("="*70 + "\n")

if __name__ == "__main__":
    main()

✓ All libraries imported successfully

     IMPROVED Pix2Pix GAN for Neovascularization Generation
               Grayscale Masks + Edge-Aware + Vessel-Focused
Device: cuda

✓ Extracted images
✓ Extracted masks

Dataset Statistics:
✓ Found 150 valid image-mask pairs
  Image size: 512x512
  Image channels: 3 (RGB fundus)
  Mask channels: 1 (GRAYSCALE vessels)

Example pair:
  Image: 06051963_.jpg
  Mask:  06051963_.png


Starting training with improved losses...


INITIALIZING IMPROVED TRAINING


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 169MB/s]



Loss Configuration:
  L1 weight: 100
  Perceptual weight: 10
  Edge weight: 50
  Focal weight: 20



...


Epoch [69/200] Summary:
  G: 30.1040 | D: 0.0079 | L1: 0.0821
  Perceptual: 1.0372 | Edge: 0.0753 | Focal: 0.0961


Epoch 70/200: 100%|██████████| 75/75 [00:46<00:00,  1.62it/s, D=0.001, Edge=0.074, G=28.452, L1=0.065]


Epoch [70/200] Summary:
  G: 30.1332 | D: 0.0046 | L1: 0.0790
  Perceptual: 1.0423 | Edge: 0.0745 | Focal: 0.0919


  ✓ Progress saved


Epoch 71/200:  49%|████▉     | 37/75 [00:23<00:23,  1.62it/s, D=0.059, Edge=0.080, G=30.247, L1=0.066]

**VESSEL MASK GENERATION**

In [ ]:
"""
Helper Script: Extract Complete Vessel Masks from Fundus Images
This script processes your fundus images and creates complete vessel segmentation masks
"""

import cv2
import numpy as np
from PIL import Image
import os
import glob
from tqdm import tqdm

def extract_vessels_from_fundus(image_path, output_path):
    """
    Extract complete vessel tree from fundus image
    Uses green channel + CLAHE + morphological operations
    """

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load: {image_path}")
        return False

    # Extract green channel (best for vessels)
    green_channel = img[:, :, 1]

    # Apply CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green_channel)

    # Invert (vessels become bright)
    inverted = cv2.bitwise_not(enhanced)

    # Morphological opening to remove noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, kernel, iterations=1)

    # Adaptive thresholding for vessels
    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=15,
        C=2
    )

    # Morphological closing to connect vessel fragments
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel2, iterations=1)

    # Remove small noise
    kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel3, iterations=1)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True

def extract_vessels_advanced(image_path, output_path):
    """
    Advanced vessel extraction using matched filtering
    Better for difficult cases
    """

    img = cv2.imread(image_path)
    if img is None:
        return False

    # Extract green channel
    green = img[:, :, 1]

    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Create matched filters for vessel detection (multiple orientations)
    kernels = []
    for angle in range(0, 180, 15):  # 12 orientations
        # Create line kernel
        kernel = np.zeros((15, 15), dtype=np.float32)
        kernel[7, :] = 1.0  # Horizontal line

        # Rotate kernel
        M = cv2.getRotationMatrix2D((7, 7), angle, 1.0)
        rotated = cv2.warpAffine(kernel, M, (15, 15))

        # Gaussian smoothing perpendicular to line
        for i in range(15):
            dist = abs(i - 7)
            rotated[i, :] *= np.exp(-(dist**2) / (2 * 1.5**2))

        rotated = rotated / rotated.sum()
        kernels.append(rotated)

    # Apply matched filters
    responses = []
    for kernel in kernels:
        response = cv2.filter2D(enhanced, -1, kernel)
        responses.append(response)

    # Take maximum response across orientations
    vessel_response = np.max(responses, axis=0)

    # Threshold
    _, binary = cv2.threshold(vessel_response, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Clean up
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=2)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True

def process_dataset(input_dir, output_dir, method='simple'):
    """
    Process entire dataset to create vessel masks

    Args:
        input_dir: Directory with fundus images
        output_dir: Directory to save vessel masks
        method: 'simple' or 'advanced'
    """

    os.makedirs(output_dir, exist_ok=True)

    # Get all images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_dir, ext)))
        image_files.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    print(f"Found {len(image_files)} images to process")
    print(f"Using {method} method")
    print(f"Output directory: {output_dir}")
    print("-" * 60)

    extract_func = extract_vessels_from_fundus if method == 'simple' else extract_vessels_advanced

    success_count = 0
    for img_path in tqdm(image_files, desc="Extracting vessels"):
        filename = os.path.basename(img_path)
        name_only = os.path.splitext(filename)[0]
        output_path = os.path.join(output_dir, f"{name_only}.png")

        if extract_func(img_path, output_path):
            success_count += 1

    print(f"\n✓ Successfully processed {success_count}/{len(image_files)} images")
    print(f"✓ Vessel masks saved to: {output_dir}")

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print(" "*15 + "Vessel Mask Extraction Tool")
    print("="*70)
    print("\nThis script extracts COMPLETE vessel trees from fundus images")
    print("to create proper training masks for Pix2Pix.\n")

    # Example 1: Process images from extracted zip
    print("Example 1: Process images from 'images/images/' directory")
    print("-" * 70)

    input_directory =  '/content/images.zip' # Your fundus images
    output_directory = 'vessel_masks/'   # Output vessel masks

    if os.path.exists(input_directory):
        print(f"Processing images from: {input_directory}")
        process_dataset(input_directory, output_directory, method='simple')
    else:
        print(f"Directory not found: {input_directory}")
        print("\nPlease update the paths below:")
        print("  input_directory = 'path/to/your/fundus/images/'")
        print("  output_directory = 'path/to/save/vessel/masks/'")

    print("\n" + "="*70)
    print("NEXT STEPS:")
    print("="*70)
    print("1. Review the generated vessel masks in:", output_directory)
    print("2. If quality is good:")
    print("   - Create masks.zip from the vessel_masks/ folder")
    print("   - Use this as your new masks.zip for Pix2Pix training")
    print("3. If quality needs improvement:")
    print("   - Try method='advanced' for better results")
    print("   - Or manually refine masks in image editor")
    print("="*70 + "\n")

    # Example 2: Process single image (for testing)
    print("\nExample 2: Test on single image")
    print("-" * 70)
    print("To test on a single image:")
    print("  extract_vessels_from_fundus('input.jpg', 'output_mask.png')")
    print("  or")
    print("  extract_vessels_advanced('input.jpg', 'output_mask.png')")
    print()


               Vessel Mask Extraction Tool

This script extracts COMPLETE vessel trees from fundus images
to create proper training masks for Pix2Pix.

Example 1: Process images from 'images/images/' directory
----------------------------------------------------------------------
Processing images from: /content/images.zip
Found 0 images to process
Using simple method
Output directory: vessel_masks/
------------------------------------------------------------


Extracting vessels: 0it [00:00, ?it/s]


✓ Successfully processed 0/0 images
✓ Vessel masks saved to: vessel_masks/

NEXT STEPS:
1. Review the generated vessel masks in: vessel_masks/
2. If quality is good:
   - Create masks.zip from the vessel_masks/ folder
   - Use this as your new masks.zip for Pix2Pix training
3. If quality needs improvement:
   - Try method='advanced' for better results
   - Or manually refine masks in image editor


Example 2: Test on single image
----------------------------------------------------------------------
To test on a single image:
  extract_vessels_from_fundus('input.jpg', 'output_mask.png')
  or
  extract_vessels_advanced('input.jpg', 'output_mask.png')



In [ ]:
# Upload a test fundus image
from google.colab import files
uploaded = files.upload()  # Upload one fundus image

# Get the filename
import os
filename = list(uploaded.keys())[0]
print(f"Processing: {filename}")

# Extract vessels
from extract_vessel_masks import extract_vessels_from_fundus
extract_vessels_from_fundus(filename, 'test_vessel_mask.png')

# Display results
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Original image
original = Image.open(filename)
axes[0].imshow(original)
axes[0].set_title('Original Fundus')
axes[0].axis('off')

# Extracted vessel mask
mask = Image.open('test_vessel_mask.png')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Extracted Vessel Mask')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Download the mask if you want
files.download('test_vessel_mask.png')

Saving Messidor-Base34-20051208_42314_0400_PP.png to Messidor-Base34-20051208_42314_0400_PP.png
Processing: Messidor-Base34-20051208_42314_0400_PP.png


ModuleNotFoundError: No module named 'extract_vessel_masks'

In [ ]:
"""
Helper Script: Extract Complete Vessel Masks from Fundus Images
This script processes your fundus images and creates complete vessel segmentation masks
"""

import cv2
import numpy as np
from PIL import Image
import os
import glob
from tqdm import tqdm

def extract_vessels_from_fundus(image_path, output_path):
    """
    Extract complete vessel tree from fundus image
    Uses green channel + CLAHE + morphological operations
    """

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load: {image_path}")
        return False

    # Extract green channel (best for vessels)
    green_channel = img[:, :, 1]

    # Apply CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green_channel)

    # Invert (vessels become bright)
    inverted = cv2.bitwise_not(enhanced)

    # Morphological opening to remove noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, kernel, iterations=1)

    # Adaptive thresholding for vessels
    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=15,
        C=2
    )

    # Morphological closing to connect vessel fragments
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel2, iterations=1)

    # Remove small noise
    kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel3, iterations=1)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True

def extract_vessels_advanced(image_path, output_path):
    """
    Advanced vessel extraction using matched filtering
    Better for difficult cases
    """

    img = cv2.imread(image_path)
    if img is None:
        return False

    # Extract green channel
    green = img[:, :, 1]

    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Create matched filters for vessel detection (multiple orientations)
    kernels = []
    for angle in range(0, 180, 15):  # 12 orientations
        # Create line kernel
        kernel = np.zeros((15, 15), dtype=np.float32)
        kernel[7, :] = 1.0  # Horizontal line

        # Rotate kernel
        M = cv2.getRotationMatrix2D((7, 7), angle, 1.0)
        rotated = cv2.warpAffine(kernel, M, (15, 15))

        # Gaussian smoothing perpendicular to line
        for i in range(15):
            dist = abs(i - 7)
            rotated[i, :] *= np.exp(-(dist**2) / (2 * 1.5**2))

        rotated = rotated / rotated.sum()
        kernels.append(rotated)

    # Apply matched filters
    responses = []
    for kernel in kernels:
        response = cv2.filter2D(enhanced, -1, kernel)
        responses.append(response)

    # Take maximum response across orientations
    vessel_response = np.max(responses, axis=0)

    # Threshold
    _, binary = cv2.threshold(vessel_response, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Clean up
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=2)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True

def process_dataset(input_dir, output_dir, method='simple'):
    """
    Process entire dataset to create vessel masks

    Args:
        input_dir: Directory with fundus images
        output_dir: Directory to save vessel masks
        method: 'simple' or 'advanced'
    """

    os.makedirs(output_dir, exist_ok=True)

    # Get all images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_dir, ext)))
        image_files.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    print(f"Found {len(image_files)} images to process")
    print(f"Using {method} method")
    print(f"Output directory: {output_dir}")
    print("-" * 60)

    extract_func = extract_vessels_from_fundus if method == 'simple' else extract_vessels_advanced

    success_count = 0
    for img_path in tqdm(image_files, desc="Extracting vessels"):
        filename = os.path.basename(img_path)
        name_only = os.path.splitext(filename)[0]
        output_path = os.path.join(output_dir, f"{name_only}.png")

        if extract_func(img_path, output_path):
            success_count += 1

    print(f"\n✓ Successfully processed {success_count}/{len(image_files)} images")
    print(f"✓ Vessel masks saved to: {output_dir}")

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print(" "*15 + "Vessel Mask Extraction Tool")
    print("="*70)
    print("\nThis script extracts COMPLETE vessel trees from fundus images")
    print("to create proper training masks for Pix2Pix.\n")

    # Auto-detect image directories
    possible_dirs = [
        'images/images/',      # Nested structure
        'images/',             # Direct structure
        'fundus_images/',
        'data/images/',
    ]

    input_directory = "/content/images.zip"
    for dir_path in possible_dirs:
        if os.path.exists(dir_path):
            # Check if directory has images
            test_files = glob.glob(os.path.join(dir_path, '*.png')) + \
                        glob.glob(os.path.join(dir_path, '*.jpg')) + \
                        glob.glob(os.path.join(dir_path, '*.jpeg'))
            if len(test_files) > 0:
                input_directory = dir_path
                break

    output_directory = 'vessel_masks/'

    if input_directory:
        print("✓ Auto-detected image directory!")
        print(f"  Input:  {input_directory}")
        print(f"  Output: {output_directory}")
        print("-" * 70)

        # Ask user for confirmation
        print("\nOptions:")
        print("  1. Process with SIMPLE method (fast, good for most cases)")
        print("  2. Process with ADVANCED method (slower, better quality)")
        print("  3. Skip automatic processing")
        print()

        # For now, auto-run with simple method
        print("Running with SIMPLE method...")
        print("-" * 70)
        process_dataset(input_directory, output_directory, method='simple')

        print("\n" + "="*70)
        print("NEXT STEPS:")
        print("="*70)
        print(f"1. Review the generated vessel masks in: {output_directory}")
        print("2. If quality is good:")
        print("   - Create masks.zip from vessel_masks/ folder:")
        print("     !zip -r masks.zip vessel_masks/")
        print("   - Use this as your new masks.zip for Pix2Pix training")
        print("3. If quality needs improvement:")
        print("   - Re-run with method='advanced'")
        print("   - Or manually refine masks in image editor")
        print("="*70 + "\n")

    else:
        print("❌ Could not find image directory automatically")
        print("\nSearched in:")
        for d in possible_dirs:
            print(f"  - {d}")
        print("\nMANUAL SETUP:")
        print("-" * 70)
        print("Please modify the script and set:")
        print()
        print("  input_directory = 'YOUR_PATH_HERE/'")
        print("  output_directory = 'vessel_masks/'")
        print("  process_dataset(input_directory, output_directory, method='simple')")
        print()
        print("Or use it programmatically:")
        print("-" * 70)

    # Example usage
    print("\nPROGRAMMATIC USAGE:")
    print("-" * 70)
    print("# Process entire directory:")
    print("from extract_vessel_masks import process_dataset")
    print("process_dataset('path/to/images/', 'vessel_masks/', method='simple')")
    print()
    print("# Process single image:")
    print("from extract_vessel_masks import extract_vessels_from_fundus")
    print("extract_vessels_from_fundus('fundus.jpg', 'vessel_mask.png')")
    print("="*70 + "\n")


               Vessel Mask Extraction Tool

This script extracts COMPLETE vessel trees from fundus images
to create proper training masks for Pix2Pix.

✓ Auto-detected image directory!
  Input:  /content/images.zip
  Output: vessel_masks/
----------------------------------------------------------------------

Options:
  1. Process with SIMPLE method (fast, good for most cases)
  2. Process with ADVANCED method (slower, better quality)
  3. Skip automatic processing

Running with SIMPLE method...
----------------------------------------------------------------------
Found 0 images to process
Using simple method
Output directory: vessel_masks/
------------------------------------------------------------


Extracting vessels: 0it [00:00, ?it/s]


✓ Successfully processed 0/0 images
✓ Vessel masks saved to: vessel_masks/

NEXT STEPS:
1. Review the generated vessel masks in: vessel_masks/
2. If quality is good:
   - Create masks.zip from vessel_masks/ folder:
     !zip -r masks.zip vessel_masks/
   - Use this as your new masks.zip for Pix2Pix training
3. If quality needs improvement:
   - Re-run with method='advanced'
   - Or manually refine masks in image editor


PROGRAMMATIC USAGE:
----------------------------------------------------------------------
# Process entire directory:
from extract_vessel_masks import process_dataset
process_dataset('path/to/images/', 'vessel_masks/', method='simple')

# Process single image:
from extract_vessel_masks import extract_vessels_from_fundus
extract_vessels_from_fundus('fundus.jpg', 'vessel_mask.png')



**WORKING VESSEL SEGMENTATION**

In [ ]:
"""
Helper Script: Extract Complete Vessel Masks from Fundus Images
This script processes your fundus images and creates complete vessel segmentation masks
"""

import cv2
import numpy as np
from PIL import Image
import os
import glob
from tqdm import tqdm
import zipfile
import shutil

def extract_vessels_from_fundus(image_path, output_path):
    """
    Extract complete vessel tree from fundus image
    Uses green channel + CLAHE + morphological operations
    """

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load: {image_path}")
        return False

    # Extract green channel (best for vessels)
    green_channel = img[:, :, 1]

    # Apply CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green_channel)

    # Invert (vessels become bright)
    inverted = cv2.bitwise_not(enhanced)

    # Morphological opening to remove noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, kernel, iterations=1)

    # Adaptive thresholding for vessels
    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=15,
        C=2
    )

    # Morphological closing to connect vessel fragments
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel2, iterations=1)

    # Remove small noise
    kernel3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel3, iterations=1)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True


def extract_vessels_advanced(image_path, output_path):
    """
    Advanced vessel extraction using matched filtering
    Better for difficult cases
    """

    img = cv2.imread(image_path)
    if img is None:
        return False

    # Extract green channel
    green = img[:, :, 1]

    # Apply CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Create matched filters for vessel detection (multiple orientations)
    kernels = []
    for angle in range(0, 180, 15):  # 12 orientations
        # Create line kernel
        kernel = np.zeros((15, 15), dtype=np.float32)
        kernel[7, :] = 1.0  # Horizontal line

        # Rotate kernel
        M = cv2.getRotationMatrix2D((7, 7), angle, 1.0)
        rotated = cv2.warpAffine(kernel, M, (15, 15))

        # Gaussian smoothing perpendicular to line
        for i in range(15):
            dist = abs(i - 7)
            rotated[i, :] *= np.exp(-(dist**2) / (2 * 1.5**2))

        rotated = rotated / rotated.sum()
        kernels.append(rotated)

    # Apply matched filters
    responses = []
    for kernel in kernels:
        response = cv2.filter2D(enhanced, -1, kernel)
        responses.append(response)

    # Take maximum response across orientations
    vessel_response = np.max(responses, axis=0)

    # Threshold
    _, binary = cv2.threshold(vessel_response, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Clean up
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_OPEN, kernel, iterations=2)

    # Save
    cv2.imwrite(output_path, cleaned)
    return True


def unzip_if_needed(path):
    """
    Check if path is a ZIP file and extract it
    Returns the directory containing the images
    """
    if path.endswith('.zip') and os.path.isfile(path):
        print(f"📦 Detected ZIP file: {path}")

        # Create extraction directory
        extract_dir = path.replace('.zip', '_extracted')

        # Remove old extraction if exists
        if os.path.exists(extract_dir):
            print(f"🗑️  Removing old extraction directory...")
            shutil.rmtree(extract_dir)

        print(f"📂 Extracting to: {extract_dir}")

        # Extract ZIP
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(f"✓ Extraction complete!")

        # Find the actual image directory (might be nested)
        # Look for directories with images
        for root, dirs, files in os.walk(extract_dir):
            # Check if this directory has image files
            image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
            if len(image_files) > 0:
                print(f"✓ Found {len(image_files)} images in: {root}")
                return root

        return extract_dir

    return path


def process_dataset(input_dir, output_dir, method='simple'):
    """
    Process entire dataset to create vessel masks

    Args:
        input_dir: Directory with fundus images (or path to ZIP file)
        output_dir: Directory to save vessel masks
        method: 'simple' or 'advanced'
    """

    # Handle ZIP files
    input_dir = unzip_if_needed(input_dir)

    os.makedirs(output_dir, exist_ok=True)

    # Get all images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_dir, ext)))
        image_files.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    print(f"\n{'='*70}")
    print(f"Found {len(image_files)} images to process")
    print(f"Using {method} method")
    print(f"Output directory: {output_dir}")
    print(f"{'='*70}\n")

    if len(image_files) == 0:
        print("❌ No images found!")
        print(f"Searched in: {input_dir}")
        print("Supported formats: .jpg, .jpeg, .png, .bmp, .tif, .tiff")
        return

    extract_func = extract_vessels_from_fundus if method == 'simple' else extract_vessels_advanced

    success_count = 0
    for img_path in tqdm(image_files, desc="Extracting vessels"):
        filename = os.path.basename(img_path)
        name_only = os.path.splitext(filename)[0]
        output_path = os.path.join(output_dir, f"{name_only}.png")

        if extract_func(img_path, output_path):
            success_count += 1

    print(f"\n{'='*70}")
    print(f"✓ Successfully processed {success_count}/{len(image_files)} images")
    print(f"✓ Vessel masks saved to: {output_dir}")
    print(f"{'='*70}\n")


# ============================================================================
# USAGE EXAMPLES
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print(" "*15 + "Vessel Mask Extraction Tool")
    print("="*70)
    print("\nThis script extracts COMPLETE vessel trees from fundus images")
    print("to create proper training masks for Pix2Pix.\n")

    # Auto-detect image directories or ZIP files
    possible_paths = [
        '/content/images.zip',      # Google Colab common location
        'images.zip',               # Current directory
        'images/images/',           # Nested structure
        'images/',                  # Direct structure
        'fundus_images/',
        'data/images/',
    ]

    input_path = None
    for path in possible_paths:
        if os.path.exists(path):
            # For directories, check if they have images
            if os.path.isdir(path):
                test_files = glob.glob(os.path.join(path, '*.png')) + \
                            glob.glob(os.path.join(path, '*.jpg')) + \
                            glob.glob(os.path.join(path, '*.jpeg'))
                if len(test_files) > 0:
                    input_path = path
                    break
            # For ZIP files, just check existence
            elif path.endswith('.zip'):
                input_path = path
                break

    output_directory = 'vessel_masks/'

    if input_path:
        print("✓ Auto-detected input!")
        print(f"  Input:  {input_path}")
        print(f"  Output: {output_directory}")
        print("="*70)

        # Ask user for confirmation
        print("\nOptions:")
        print("  1. Process with SIMPLE method (fast, good for most cases)")
        print("  2. Process with ADVANCED method (slower, better quality)")
        print("  3. Skip automatic processing")
        print()

        # For now, auto-run with simple method
        print("Running with SIMPLE method...")
        print("="*70)
        process_dataset(input_path, output_directory, method='simple')

        print("\n" + "="*70)
        print("NEXT STEPS:")
        print("="*70)
        print(f"1. Review the generated vessel masks in: {output_directory}")
        print("2. If quality is good:")
        print("   - Create masks.zip from vessel_masks/ folder:")
        print("     !zip -r masks.zip vessel_masks/")
        print("   - Use this as your new masks.zip for Pix2Pix training")
        print("3. If quality needs improvement:")
        print("   - Re-run with method='advanced'")
        print("   - Or manually refine masks in image editor")
        print("="*70 + "\n")

    else:
        print("❌ Could not find image directory or ZIP file automatically")
        print("\nSearched in:")
        for p in possible_paths:
            print(f"  - {p}")
        print("\nMANUAL SETUP:")
        print("="*70)
        print("Please modify the script and set:")
        print()
        print("  input_path = 'YOUR_PATH_HERE.zip'  # or directory path")
        print("  output_directory = 'vessel_masks/'")
        print("  process_dataset(input_path, output_directory, method='simple')")
        print()
        print("Or use it programmatically:")
        print("="*70)

    # Example usage
    print("\nPROGRAMMATIC USAGE:")
    print("="*70)
    print("# Process ZIP file or directory:")
    print("from extract_vessel_masks import process_dataset")
    print("process_dataset('images.zip', 'vessel_masks/', method='simple')")
    print()
    print("# Process single image:")
    print("from extract_vessel_masks import extract_vessels_from_fundus")
    print("extract_vessels_from_fundus('fundus.jpg', 'vessel_mask.png')")
    print("="*70 + "\n")


               Vessel Mask Extraction Tool

This script extracts COMPLETE vessel trees from fundus images
to create proper training masks for Pix2Pix.

✓ Auto-detected input!
  Input:  /content/images.zip
  Output: vessel_masks/

Options:
  1. Process with SIMPLE method (fast, good for most cases)
  2. Process with ADVANCED method (slower, better quality)
  3. Skip automatic processing

Running with SIMPLE method...
📦 Detected ZIP file: /content/images.zip
📂 Extracting to: /content/images_extracted
✓ Extraction complete!
✓ Found 150 images in: /content/images_extracted/images

Found 150 images to process
Using simple method
Output directory: vessel_masks/



Extracting vessels: 100%|██████████| 150/150 [00:18<00:00,  8.29it/s]


✓ Successfully processed 150/150 images
✓ Vessel masks saved to: vessel_masks/


NEXT STEPS:
1. Review the generated vessel masks in: vessel_masks/
2. If quality is good:
   - Create masks.zip from vessel_masks/ folder:
     !zip -r masks.zip vessel_masks/
   - Use this as your new masks.zip for Pix2Pix training
3. If quality needs improvement:
   - Re-run with method='advanced'
   - Or manually refine masks in image editor


PROGRAMMATIC USAGE:
# Process ZIP file or directory:
from extract_vessel_masks import process_dataset
process_dataset('images.zip', 'vessel_masks/', method='simple')

# Process single image:
from extract_vessel_masks import extract_vessels_from_fundus
extract_vessels_from_fundus('fundus.jpg', 'vessel_mask.png')



In [ ]:
"""
Retinal Vessel Mask Extraction for Pix2Pix Training
Generates binary masks: WHITE vessels on BLACK background
"""

import cv2
import numpy as np
from PIL import Image
import os
import glob
from tqdm import tqdm
import zipfile
import shutil

def extract_vessels_for_pix2pix(image_path, output_path, method='adaptive'):
    """
    Extract clean vessel mask for Pix2Pix training
    Returns: Binary mask with WHITE vessels (255) on BLACK background (0)

    Methods:
        'adaptive': Best for most fundus images (recommended)
        'otsu': Good for high contrast images
        'frangi': Advanced vessel enhancement (slower but better)
    """

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load: {image_path}")
        return False

    # Extract green channel (vessels are most visible here)
    green = img[:, :, 1]

    # Apply CLAHE for better contrast
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    if method == 'frangi':
        # Advanced Frangi filter for vessel enhancement
        vessel_mask = apply_frangi_filter(enhanced)
    elif method == 'otsu':
        # Simple Otsu thresholding
        inverted = 255 - enhanced
        _, vessel_mask = cv2.threshold(inverted, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:  # adaptive (default)
        # Invert so vessels are bright
        inverted = 255 - enhanced

        # Apply Gaussian blur to reduce noise
        blurred = cv2.GaussianBlur(inverted, (5, 5), 0)

        # Adaptive thresholding
        vessel_mask = cv2.adaptiveThreshold(
            blurred,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            blockSize=11,  # Smaller = captures finer vessels
            C=2
        )

    # Post-processing to clean up the mask

    # 1. Remove small noise (small white dots)
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(vessel_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # 2. Connect nearby vessel fragments
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # 3. Optional: Skeletonize to get thin vessels (comment out if you want thicker vessels)
    # connected = skeletonize(connected)

    # Ensure binary: 0 or 255 only
    _, final_mask = cv2.threshold(connected, 127, 255, cv2.THRESH_BINARY)

    # Save
    cv2.imwrite(output_path, final_mask)
    return True


def apply_frangi_filter(image):
    """
    Apply Frangi vesselness filter for better vessel detection
    This enhances tubular structures (vessels)
    """
    # Normalize image
    normalized = image.astype(np.float32) / 255.0

    # Apply Frangi filter at multiple scales
    scales = [1, 2, 3, 4]  # Different vessel widths
    responses = []

    for scale in scales:
        sigma = scale

        # Compute Hessian matrix components
        Ixx = cv2.Sobel(normalized, cv2.CV_64F, 2, 0, ksize=2*int(3*sigma)+1)
        Iyy = cv2.Sobel(normalized, cv2.CV_64F, 0, 2, ksize=2*int(3*sigma)+1)
        Ixy = cv2.Sobel(normalized, cv2.CV_64F, 1, 1, ksize=2*int(3*sigma)+1)

        # Smooth
        Ixx = cv2.GaussianBlur(Ixx, (0, 0), sigma)
        Iyy = cv2.GaussianBlur(Iyy, (0, 0), sigma)
        Ixy = cv2.GaussianBlur(Ixy, (0, 0), sigma)

        # Compute eigenvalues at each pixel
        vesselness = np.zeros_like(normalized)
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                hessian = np.array([[Ixx[i,j], Ixy[i,j]],
                                   [Ixy[i,j], Iyy[i,j]]])
                eigenvalues = np.linalg.eigvalsh(hessian)
                eigenvalues = np.sort(np.abs(eigenvalues))

                # Frangi's vesselness measure
                if eigenvalues[1] < 0:  # Dark vessels on bright background
                    Rb = eigenvalues[0] / (eigenvalues[1] + 1e-10)
                    S = np.sqrt(eigenvalues[0]**2 + eigenvalues[1]**2)
                    vesselness[i,j] = np.exp(-Rb**2 / (2 * 0.5**2)) * (1 - np.exp(-S**2 / (2 * 15**2)))

        responses.append(vesselness * (sigma**2))

    # Take maximum response across scales
    final_response = np.max(responses, axis=0)

    # Threshold
    _, binary = cv2.threshold((final_response * 255).astype(np.uint8), 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary


def skeletonize(binary_image):
    """
    Morphological skeletonization to get thin vessel centerlines
    """
    skeleton = np.zeros_like(binary_image)
    element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))

    while True:
        opened = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, element)
        temp = cv2.subtract(binary_image, opened)
        eroded = cv2.erode(binary_image, element)
        skeleton = cv2.bitwise_or(skeleton, temp)
        binary_image = eroded.copy()

        if cv2.countNonZero(binary_image) == 0:
            break

    return skeleton


def unzip_if_needed(path):
    """
    Check if path is a ZIP file and extract it
    Returns the directory containing the images
    """
    if path.endswith('.zip') and os.path.isfile(path):
        print(f"📦 Detected ZIP file: {path}")

        # Create extraction directory
        extract_dir = path.replace('.zip', '_extracted')

        # Remove old extraction if exists
        if os.path.exists(extract_dir):
            print(f"🗑️  Removing old extraction directory...")
            shutil.rmtree(extract_dir)

        print(f"📂 Extracting to: {extract_dir}")

        # Extract ZIP
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(f"✓ Extraction complete!")

        # Find the actual image directory (might be nested)
        for root, dirs, files in os.walk(extract_dir):
            image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
            if len(image_files) > 0:
                print(f"✓ Found {len(image_files)} images in: {root}")
                return root

        return extract_dir

    return path


def create_preview_comparison(image_path, mask_path, output_path):
    """
    Create side-by-side comparison: original | mask | overlay
    """
    original = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if original is None or mask is None:
        return

    # Resize to same height
    h = original.shape[0]
    mask_resized = cv2.resize(mask, (original.shape[1], h))

    # Create colored overlay (vessels in green on original)
    overlay = original.copy()
    overlay[mask_resized > 127] = [0, 255, 0]  # Green vessels
    overlay = cv2.addWeighted(original, 0.7, overlay, 0.3, 0)

    # Stack horizontally
    mask_colored = cv2.cvtColor(mask_resized, cv2.COLOR_GRAY2BGR)
    comparison = np.hstack([original, mask_colored, overlay])

    cv2.imwrite(output_path, comparison)


def process_dataset(input_dir, output_dir, method='adaptive', create_previews=True):
    """
    Process entire dataset to create vessel masks for Pix2Pix

    Args:
        input_dir: Directory with fundus images (or path to ZIP file)
        output_dir: Directory to save vessel masks
        method: 'adaptive' (recommended), 'otsu', or 'frangi'
        create_previews: If True, creates preview images for quality check
    """

    # Handle ZIP files
    input_dir = unzip_if_needed(input_dir)

    os.makedirs(output_dir, exist_ok=True)

    if create_previews:
        preview_dir = os.path.join(output_dir, '_previews')
        os.makedirs(preview_dir, exist_ok=True)

    # Get all images
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
    image_files = []
    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_dir, ext)))
        image_files.extend(glob.glob(os.path.join(input_dir, ext.upper())))

    print(f"\n{'='*70}")
    print(f"Found {len(image_files)} images to process")
    print(f"Method: {method.upper()}")
    print(f"Output: {output_dir}")
    print(f"{'='*70}\n")

    if len(image_files) == 0:
        print("❌ No images found!")
        print(f"Searched in: {input_dir}")
        print("Supported formats: .jpg, .jpeg, .png, .bmp, .tif, .tiff")
        return

    success_count = 0
    preview_count = 0

    for idx, img_path in enumerate(tqdm(image_files, desc="Extracting vessel masks")):
        filename = os.path.basename(img_path)
        name_only = os.path.splitext(filename)[0]

        # Output mask path
        mask_path = os.path.join(output_dir, f"{name_only}.png")

        # Extract vessels
        if extract_vessels_for_pix2pix(img_path, mask_path, method=method):
            success_count += 1

            # Create preview for first 5 images
            if create_previews and preview_count < 5:
                preview_path = os.path.join(preview_dir, f"preview_{name_only}.png")
                create_preview_comparison(img_path, mask_path, preview_path)
                preview_count += 1

    print(f"\n{'='*70}")
    print(f"✅ Successfully processed {success_count}/{len(image_files)} images")
    print(f"✅ Vessel masks saved to: {output_dir}")
    if create_previews:
        print(f"✅ Preview images saved to: {preview_dir}")
    print(f"{'='*70}\n")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    print("\n" + "="*70)
    print(" "*10 + "Retinal Vessel Mask Extractor for Pix2Pix")
    print("="*70)
    print("\nGenerates binary masks: WHITE vessels on BLACK background")
    print("Perfect for training Pix2Pix models!\n")

    # Auto-detect image sources
    possible_paths = [
        '/content/images.zip',      # Google Colab
        'images.zip',
        'images/images/',
        'images/',
        'fundus_images/',
        'data/images/',
    ]

    input_path = None
    for path in possible_paths:
        if os.path.exists(path):
            if os.path.isdir(path):
                test_files = glob.glob(os.path.join(path, '*.png')) + \
                            glob.glob(os.path.join(path, '*.jpg')) + \
                            glob.glob(os.path.join(path, '*.jpeg'))
                if len(test_files) > 0:
                    input_path = path
                    break
            elif path.endswith('.zip'):
                input_path = path
                break

    output_directory = 'vessel_masks/'

    if input_path:
        print("✅ Auto-detected input!")
        print(f"  Input:  {input_path}")
        print(f"  Output: {output_directory}")
        print("="*70)

        print("\n📋 AVAILABLE METHODS:")
        print("  • adaptive  - Best for most cases (RECOMMENDED)")
        print("  • otsu      - Simple, fast, good for high contrast")
        print("  • frangi    - Advanced, slower, best quality")
        print()
        print("🚀 Running with ADAPTIVE method...")
        print("="*70)

        # Process with adaptive method (best balance of speed and quality)
        process_dataset(input_path, output_directory, method='adaptive', create_previews=True)

        print("\n" + "="*70)
        print("📝 NEXT STEPS FOR PIX2PIX TRAINING:")
        print("="*70)
        print(f"1. ✅ Check preview images in: {output_directory}_previews/")
        print("   (First 5 samples showing: original | mask | overlay)")
        print()
        print("2. If masks look good (white vessels, clean, connected):")
        print("   📦 Create masks.zip:")
        print("      !cd vessel_masks && zip -r ../masks.zip *.png")
        print()
        print("3. For Pix2Pix training, you need:")
        print("   • images.zip (your fundus images) ✓")
        print("   • masks.zip (generated vessel masks) ← Use this!")
        print()
        print("4. If quality needs improvement:")
        print("   • Try method='frangi' for better vessel detection")
        print("   • Adjust blockSize parameter in adaptive method")
        print("="*70 + "\n")

    else:
        print("❌ Could not auto-detect images")
        print("\nSearched in:")
        for p in possible_paths:
            print(f"  - {p}")
        print("\n" + "="*70)
        print("MANUAL USAGE:")
        print("="*70)
        print("from extract_vessel_masks import process_dataset")
        print("process_dataset('images.zip', 'vessel_masks/', method='adaptive')")
        print("="*70 + "\n")


          Retinal Vessel Mask Extractor for Pix2Pix

Generates binary masks: WHITE vessels on BLACK background
Perfect for training Pix2Pix models!

✅ Auto-detected input!
  Input:  /content/images.zip
  Output: vessel_masks/

📋 AVAILABLE METHODS:
  • adaptive  - Best for most cases (RECOMMENDED)
  • otsu      - Simple, fast, good for high contrast
  • frangi    - Advanced, slower, best quality

🚀 Running with ADAPTIVE method...
📦 Detected ZIP file: /content/images.zip
📂 Extracting to: /content/images_extracted
✓ Extraction complete!
✓ Found 150 images in: /content/images_extracted/images

Found 150 images to process
Method: ADAPTIVE
Output: vessel_masks/



Extracting vessel masks: 100%|██████████| 150/150 [00:22<00:00,  6.59it/s]


✅ Successfully processed 150/150 images
✅ Vessel masks saved to: vessel_masks/
✅ Preview images saved to: vessel_masks/_previews


📝 NEXT STEPS FOR PIX2PIX TRAINING:
1. ✅ Check preview images in: vessel_masks/_previews/
   (First 5 samples showing: original | mask | overlay)

2. If masks look good (white vessels, clean, connected):
   📦 Create masks.zip:
      !cd vessel_masks && zip -r ../masks.zip *.png

3. For Pix2Pix training, you need:
   • images.zip (your fundus images) ✓
   • masks.zip (generated vessel masks) ← Use this!

4. If quality needs improvement:
   • Try method='frangi' for better vessel detection
   • Adjust blockSize parameter in adaptive method



**VESSEL EXTRACTION (BINARY MASKS)**

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install opencv-python-headless numpy Pillow tqdm


# ============================================================
# CELL 2: Upload images.zip
# ============================================================
from google.colab import files
uploaded = files.upload()
print("✅ Upload complete!")


# ============================================================
# CELL 3: Unzip images.zip
# ============================================================
import zipfile
import os

# Unzip the uploaded file
zip_filename = list(uploaded.keys())[0]
print(f"📦 Unzipping: {zip_filename}")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/uploaded_data/')

# Auto-detect the images folder
# Your structure: images.zip -> images/ -> *.jpg/png
image_dir = None
for root, dirs, file_list in os.walk('/content/uploaded_data/'):
    # Skip __MACOSX and hidden directories
    if '__MACOSX' in root or '/.' in root:
        continue
    image_files_in_dir = [f for f in file_list
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                          and not f.startswith('.')]
    if len(image_files_in_dir) > 0:
        image_dir = root
        break

if image_dir:
    all_images = [f for f in os.listdir(image_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                  and not f.startswith('.')]
    print(f"✅ Found {len(all_images)} images in: {image_dir}")
    print(f"📸 Sample files: {all_images[:5]}")
else:
    print("❌ No images found! Check your zip structure.")
    print("Expected: images.zip -> images/ -> image1.jpg, image2.jpg, ...")


# ============================================================
# CELL 4: Vessel Extraction Functions
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

def extract_vessels(image_path, method='simple'):
    """
    Extract retinal vessels from a fundus image.
    Returns a binary mask: white (255) vessels on black (0) background.

    Args:
        image_path: Path to fundus image
        method: 'simple' or 'advanced'

    Returns:
        Binary mask (numpy array) or None if failed
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"  ⚠️ Failed to load: {image_path}")
        return None

    # Get image dimensions for adaptive parameters
    h, w = img.shape[:2]

    # ---- Step 1: Extract green channel (best vessel contrast) ----
    green = img[:, :, 1]

    # ---- Step 2: Create FOV (Field of View) mask ----
    # This blacks out everything outside the retinal area
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_for_mask = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_for_mask, 15, 255, cv2.THRESH_BINARY)
    # Erode FOV mask to remove edge artifacts
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    if method == 'advanced':
        return _extract_advanced(green, fov_mask, w, h)
    else:
        return _extract_simple(green, fov_mask, w, h)


def _extract_simple(green, fov_mask, w, h):
    """
    Simple method: CLAHE → Invert → Top-hat → Adaptive Threshold → Cleanup
    Fast and works well for most fundus images.
    """
    # ---- CLAHE Enhancement ----
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # ---- Invert (vessels become bright) ----
    inverted = cv2.bitwise_not(enhanced)

    # ---- Top-hat transform (removes uneven background illumination) ----
    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, kernel_tophat)

    # ---- Second CLAHE on top-hat result ----
    clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    enhanced2 = clahe2.apply(tophat)

    # ---- Morphological opening (remove small noise) ----
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # ---- Adaptive thresholding ----
    # Block size should be odd and reasonable for image size
    block_size = max(11, min(31, (min(w, h) // 40) | 1))
    if block_size % 2 == 0:
        block_size += 1

    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=block_size,
        C=3
    )

    # ---- Morphological closing (connect vessel fragments) ----
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # ---- Remove small connected components (noise) ----
    min_size = max(15, int(w * h * 0.00005))
    cleaned = _remove_small_components(closed, min_size)

    # ---- Final morphological opening ----
    kernel_final = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_final, iterations=1)

    # ---- Apply FOV mask ----
    cleaned[fov_mask == 0] = 0

    # ---- Ensure strict binary ----
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    return result


def _extract_advanced(green, fov_mask, w, h):
    """
    Advanced method: Matched filtering with multiple orientations.
    Better for noisy or low-quality images.
    """
    # ---- CLAHE Enhancement ----
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # ---- Matched Filter Bank (12 orientations) ----
    ksize = 15
    num_orientations = 12
    center = ksize // 2
    responses = []

    for oi in range(num_orientations):
        angle = (oi * 180.0) / num_orientations
        angle_rad = np.radians(angle)

        # Create oriented Gaussian line kernel
        kernel = np.zeros((ksize, ksize), dtype=np.float64)
        cos_a = np.cos(angle_rad)
        sin_a = np.sin(angle_rad)
        sigma = 1.5

        for y in range(ksize):
            for x in range(ksize):
                dx = x - center
                dy = y - center
                along = dx * cos_a + dy * sin_a
                perp = -dx * sin_a + dy * cos_a
                if abs(along) <= center:
                    kernel[y, x] = np.exp(-(perp ** 2) / (2 * sigma ** 2))

        # Normalize
        k_sum = kernel.sum()
        if k_sum > 0:
            kernel /= k_sum

        # Make zero-mean (suppresses uniform background)
        kernel -= kernel.mean()

        # Apply filter
        response = cv2.filter2D(enhanced.astype(np.float64), -1, kernel)
        responses.append(response)

    # ---- Maximum response across all orientations ----
    vessel_response = np.max(responses, axis=0)

    # ---- Normalize to 0-255 range (only within FOV) ----
    fov_pixels = vessel_response[fov_mask > 0]
    if len(fov_pixels) > 0:
        min_r = fov_pixels.min()
        max_r = fov_pixels.max()
        range_r = max_r - min_r if max_r > min_r else 1
        normalized = np.clip(((vessel_response - min_r) / range_r) * 255, 0, 255).astype(np.uint8)
    else:
        normalized = np.zeros_like(vessel_response, dtype=np.uint8)

    normalized[fov_mask == 0] = 0

    # ---- Otsu threshold ----
    _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # ---- Morphological cleanup ----
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open, iterations=1)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # ---- Remove small components ----
    min_size = max(20, int(w * h * 0.00008))
    cleaned = _remove_small_components(cleaned, min_size)

    # ---- Apply FOV mask ----
    cleaned[fov_mask == 0] = 0

    # ---- Ensure strict binary ----
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    return result


def _remove_small_components(binary, min_size):
    """Remove connected components smaller than min_size pixels."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    result = np.zeros_like(binary)
    for i in range(1, num_labels):  # Skip background (label 0)
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            result[labels == i] = 255
    return result


# ============================================================
# CELL 5: Process All Images & Create Vessel Masks
# ============================================================
import glob

# Output directory for vessel masks
output_dir = '/content/vessel_masks/'
os.makedirs(output_dir, exist_ok=True)

# Choose method: 'simple' (fast) or 'advanced' (better quality, slower)
METHOD = 'simple'  # <-- Change to 'advanced' if needed

print(f"🔬 Method: {METHOD}")
print(f"📂 Input:  {image_dir}")
print(f"📂 Output: {output_dir}")
print("=" * 60)

# Get all image files
image_files = sorted([
    os.path.join(image_dir, f) for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
    and not f.startswith('.')
])

print(f"🖼️  Found {len(image_files)} images to process\n")

success_count = 0
fail_count = 0

for img_path in tqdm(image_files, desc="Extracting vessels"):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    output_path = os.path.join(output_dir, f"{name_only}.png")

    mask = extract_vessels(img_path, method=METHOD)

    if mask is not None:
        cv2.imwrite(output_path, mask)
        success_count += 1
    else:
        fail_count += 1

print(f"\n{'=' * 60}")
print(f"✅ Successfully processed: {success_count}/{len(image_files)}")
if fail_count > 0:
    print(f"❌ Failed: {fail_count}")
print(f"📁 Vessel masks saved to: {output_dir}")
print(f"{'=' * 60}")


# ============================================================
# CELL 6: Preview Some Results (Side by Side)
# ============================================================
import matplotlib.pyplot as plt

# Show up to 5 random samples
sample_files = image_files[:5]  # First 5 images

fig, axes = plt.subplots(len(sample_files), 2, figsize=(12, 5 * len(sample_files)))

if len(sample_files) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_files):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    mask_path = os.path.join(output_dir, f"{name_only}.png")

    # Load original
    original = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    # Load mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Plot
    axes[idx][0].imshow(original_rgb)
    axes[idx][0].set_title(f'Original: {filename}', fontsize=10)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(mask, cmap='gray', vmin=0, vmax=255)
    axes[idx][1].set_title(f'Vessel Mask: {name_only}.png', fontsize=10)
    axes[idx][1].axis('off')

plt.suptitle('Fundus Images → Vessel Masks (White on Black)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show pixel value stats to confirm binary
sample_mask = cv2.imread(os.path.join(output_dir, os.listdir(output_dir)[0]), cv2.IMREAD_GRAYSCALE)
unique_vals = np.unique(sample_mask)
print(f"\n🔍 Pixel value check: unique values = {unique_vals}")
print(f"   ✅ {'Binary confirmed! Only 0 and 255.' if set(unique_vals).issubset({0, 255}) else '⚠️ Not strictly binary!'}")


# ============================================================
# CELL 7: Zip & Download Vessel Masks
# ============================================================
import shutil

# Create zip of vessel masks
zip_output = '/content/vessel_masks'
shutil.make_archive(zip_output, 'zip', '/content/', 'vessel_masks')

print(f"📦 Created: vessel_masks.zip")
print(f"   Contains {len(os.listdir(output_dir))} vessel mask images")
print(f"   Format: White vessels (255) on black background (0)")
print()

# Auto-download
from google.colab import files
files.download('/content/vessel_masks.zip')
print("⬇️  Download started!")


# ============================================================
# CELL 8 (OPTIONAL): If simple method quality is not good,
#                     re-run with advanced method
# ============================================================

# Uncomment below to re-run with advanced method:

# METHOD = 'advanced'
# output_dir_adv = '/content/vessel_masks_advanced/'
# os.makedirs(output_dir_adv, exist_ok=True)
#
# for img_path in tqdm(image_files, desc="Advanced extraction"):
#     filename = os.path.basename(img_path)
#     name_only = os.path.splitext(filename)[0]
#     output_path = os.path.join(output_dir_adv, f"{name_only}.png")
#     mask = extract_vessels(img_path, method='advanced')
#     if mask is not None:
#         cv2.imwrite(output_path, mask)
#
# shutil.make_archive('/content/vessel_masks_advanced', 'zip', '/content/', 'vessel_masks_advanced')
# files.download('/content/vessel_masks_advanced.zip')

KeyboardInterrupt: 

**CORRECT AND FINAL WORKING CODE FOR VESSEL BINARY MASKS**

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install opencv-python-headless numpy Pillow tqdm


# ============================================================
# CELL 2: Unzip images.zip (already uploaded in Colab)
# ============================================================
import zipfile
import os

# Path to your already-uploaded images.zip
zip_path = '/content/images.zip'

if not os.path.exists(zip_path):
    print(f"❌ File not found: {zip_path}")
    print("   Make sure images.zip is uploaded to /content/ in Colab")
else:
    print(f"📦 Found: {zip_path}")
    print(f"   Size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

    # Unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/uploaded_data/')
    print("✅ Unzipped successfully!")

    # Auto-detect the images folder
    # Your structure: images.zip -> images/ -> *.jpg/png
    image_dir = None
    for root, dirs, file_list in os.walk('/content/uploaded_data/'):
        # Skip __MACOSX and hidden directories
        if '__MACOSX' in root or '/.' in root:
            continue
        image_files_in_dir = [f for f in file_list
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                              and not f.startswith('.')]
        if len(image_files_in_dir) > 0:
            image_dir = root
            break

    if image_dir:
        all_images = [f for f in os.listdir(image_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                      and not f.startswith('.')]
        print(f"✅ Found {len(all_images)} images in: {image_dir}")
        print(f"📸 Sample files: {all_images[:5]}")
    else:
        print("❌ No images found! Check your zip structure.")
        print("Expected: images.zip -> images/ -> image1.jpg, image2.jpg, ...")
        # List what was extracted
        print("\nExtracted contents:")
        for root, dirs, files in os.walk('/content/uploaded_data/'):
            level = root.replace('/content/uploaded_data/', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for file in files[:10]:
                print(f"{subindent}{file}")
            if len(files) > 10:
                print(f"{subindent}... and {len(files)-10} more files")


# ============================================================
# CELL 3: Vessel Extraction Functions
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

def extract_vessels(image_path, method='simple'):
    """
    Extract retinal vessels from a fundus image.
    Returns a binary mask: white (255) vessels on black (0) background.

    Args:
        image_path: Path to fundus image
        method: 'simple' or 'advanced'

    Returns:
        Binary mask (numpy array) or None if failed
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"  ⚠️ Failed to load: {image_path}")
        return None

    # Get image dimensions for adaptive parameters
    h, w = img.shape[:2]

    # ---- Step 1: Extract green channel (best vessel contrast) ----
    green = img[:, :, 1]

    # ---- Step 2: Create FOV (Field of View) mask ----
    # This blacks out everything outside the retinal area
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_for_mask = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_for_mask, 15, 255, cv2.THRESH_BINARY)
    # Erode FOV mask to remove edge artifacts
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    if method == 'advanced':
        return _extract_advanced(green, fov_mask, w, h)
    else:
        return _extract_simple(green, fov_mask, w, h)


def _extract_simple(green, fov_mask, w, h):
    """
    Simple method: CLAHE → Invert → Top-hat → Adaptive Threshold → Cleanup
    Fast and works well for most fundus images.
    """
    # ---- CLAHE Enhancement ----
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # ---- Invert (vessels become bright) ----
    inverted = cv2.bitwise_not(enhanced)

    # ---- Top-hat transform (removes uneven background illumination) ----
    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, kernel_tophat)

    # ---- Second CLAHE on top-hat result ----
    clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    enhanced2 = clahe2.apply(tophat)

    # ---- Morphological opening (remove small noise) ----
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # ---- Adaptive thresholding ----
    # Block size should be odd and reasonable for image size
    block_size = max(11, min(31, (min(w, h) // 40) | 1))
    if block_size % 2 == 0:
        block_size += 1

    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=block_size,
        C=3
    )

    # ---- Morphological closing (connect vessel fragments) ----
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # ---- Remove small connected components (noise) ----
    min_size = max(15, int(w * h * 0.00005))
    cleaned = _remove_small_components(closed, min_size)

    # ---- Final morphological opening ----
    kernel_final = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_final, iterations=1)

    # ---- Apply FOV mask ----
    cleaned[fov_mask == 0] = 0

    # ---- Ensure strict binary ----
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    return result


def _extract_advanced(green, fov_mask, w, h):
    """
    Advanced method: Matched filtering with multiple orientations.
    Better for noisy or low-quality images.
    """
    # ---- CLAHE Enhancement ----
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # ---- Matched Filter Bank (12 orientations) ----
    ksize = 15
    num_orientations = 12
    center = ksize // 2
    responses = []

    for oi in range(num_orientations):
        angle = (oi * 180.0) / num_orientations
        angle_rad = np.radians(angle)

        # Create oriented Gaussian line kernel
        kernel = np.zeros((ksize, ksize), dtype=np.float64)
        cos_a = np.cos(angle_rad)
        sin_a = np.sin(angle_rad)
        sigma = 1.5

        for y in range(ksize):
            for x in range(ksize):
                dx = x - center
                dy = y - center
                along = dx * cos_a + dy * sin_a
                perp = -dx * sin_a + dy * cos_a
                if abs(along) <= center:
                    kernel[y, x] = np.exp(-(perp ** 2) / (2 * sigma ** 2))

        # Normalize
        k_sum = kernel.sum()
        if k_sum > 0:
            kernel /= k_sum

        # Make zero-mean (suppresses uniform background)
        kernel -= kernel.mean()

        # Apply filter
        response = cv2.filter2D(enhanced.astype(np.float64), -1, kernel)
        responses.append(response)

    # ---- Maximum response across all orientations ----
    vessel_response = np.max(responses, axis=0)

    # ---- Normalize to 0-255 range (only within FOV) ----
    fov_pixels = vessel_response[fov_mask > 0]
    if len(fov_pixels) > 0:
        min_r = fov_pixels.min()
        max_r = fov_pixels.max()
        range_r = max_r - min_r if max_r > min_r else 1
        normalized = np.clip(((vessel_response - min_r) / range_r) * 255, 0, 255).astype(np.uint8)
    else:
        normalized = np.zeros_like(vessel_response, dtype=np.uint8)

    normalized[fov_mask == 0] = 0

    # ---- Otsu threshold ----
    _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # ---- Morphological cleanup ----
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open, iterations=1)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # ---- Remove small components ----
    min_size = max(20, int(w * h * 0.00008))
    cleaned = _remove_small_components(cleaned, min_size)

    # ---- Apply FOV mask ----
    cleaned[fov_mask == 0] = 0

    # ---- Ensure strict binary ----
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    return result


def _remove_small_components(binary, min_size):
    """Remove connected components smaller than min_size pixels."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    result = np.zeros_like(binary)
    for i in range(1, num_labels):  # Skip background (label 0)
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            result[labels == i] = 255
    return result


# ============================================================
# CELL 4: Process All Images & Create Vessel Masks
# ============================================================
import glob

# Output directory for vessel masks
output_dir = '/content/vessel_masks/'
os.makedirs(output_dir, exist_ok=True)

# Choose method: 'simple' (fast) or 'advanced' (better quality, slower)
METHOD = 'simple'  # <-- Change to 'advanced' if needed

print(f"🔬 Method: {METHOD}")
print(f"📂 Input:  {image_dir}")
print(f"📂 Output: {output_dir}")
print("=" * 60)

# Get all image files
image_files = sorted([
    os.path.join(image_dir, f) for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
    and not f.startswith('.')
])

print(f"🖼️  Found {len(image_files)} images to process\n")

success_count = 0
fail_count = 0

for img_path in tqdm(image_files, desc="Extracting vessels"):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    output_path = os.path.join(output_dir, f"{name_only}.png")

    mask = extract_vessels(img_path, method=METHOD)

    if mask is not None:
        cv2.imwrite(output_path, mask)
        success_count += 1
    else:
        fail_count += 1

print(f"\n{'=' * 60}")
print(f"✅ Successfully processed: {success_count}/{len(image_files)}")
if fail_count > 0:
    print(f"❌ Failed: {fail_count}")
print(f"📁 Vessel masks saved to: {output_dir}")
print(f"{'=' * 60}")


# ============================================================
# CELL 5: Preview Some Results (Side by Side)
# ============================================================
import matplotlib.pyplot as plt

# Show up to 5 random samples
sample_files = image_files[:5]  # First 5 images

fig, axes = plt.subplots(len(sample_files), 2, figsize=(12, 5 * len(sample_files)))

if len(sample_files) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_files):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    mask_path = os.path.join(output_dir, f"{name_only}.png")

    # Load original
    original = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    # Load mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Plot
    axes[idx][0].imshow(original_rgb)
    axes[idx][0].set_title(f'Original: {filename}', fontsize=10)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(mask, cmap='gray', vmin=0, vmax=255)
    axes[idx][1].set_title(f'Vessel Mask: {name_only}.png', fontsize=10)
    axes[idx][1].axis('off')

plt.suptitle('Fundus Images → Vessel Masks (White on Black)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show pixel value stats to confirm binary
sample_mask = cv2.imread(os.path.join(output_dir, os.listdir(output_dir)[0]), cv2.IMREAD_GRAYSCALE)
unique_vals = np.unique(sample_mask)
print(f"\n🔍 Pixel value check: unique values = {unique_vals}")
print(f"   ✅ {'Binary confirmed! Only 0 and 255.' if set(unique_vals).issubset({0, 255}) else '⚠️ Not strictly binary!'}")


# ============================================================
# CELL 6: Zip & Download Vessel Masks
# ============================================================
import shutil

# Create zip of vessel masks
zip_output = '/content/vessel_masks'
shutil.make_archive(zip_output, 'zip', '/content/', 'vessel_masks')

print(f"📦 Created: vessel_masks.zip")
print(f"   Contains {len(os.listdir(output_dir))} vessel mask images")
print(f"   Format: White vessels (255) on black background (0)")
print()

# Auto-download
from google.colab import files
files.download('/content/vessel_masks.zip')
print("⬇️  Download started!")


# ============================================================
# CELL 7 (OPTIONAL): Re-run with ADVANCED method if needed
# ============================================================

# Uncomment all lines below to re-run with advanced matched filtering:

# output_dir_adv = '/content/vessel_masks_advanced/'
# os.makedirs(output_dir_adv, exist_ok=True)
#
# print("🔬 Re-running with ADVANCED method...")
# print("=" * 60)
#
# for img_path in tqdm(image_files, desc="Advanced extraction"):
#     filename = os.path.basename(img_path)
#     name_only = os.path.splitext(filename)[0]
#     output_path = os.path.join(output_dir_adv, f"{name_only}.png")
#     mask = extract_vessels(img_path, method='advanced')
#     if mask is not None:
#         cv2.imwrite(output_path, mask)
#
# shutil.make_archive('/content/vessel_masks_advanced', 'zip', '/content/', 'vessel_masks_advanced')
# files.download('/content/vessel_masks_advanced.zip')
# print("⬇️  Advanced masks downloaded!")

Output hidden; open in https://colab.research.google.com to view.

**SOFTENING MASK CODE**

**IMAGE SOFTENING CORRECT CODE**

In [ ]:
"""
SIMPLE MASK SOFTENING - GAUSSIAN BLUR ONLY
No transformations, no inversions - just gentle blur
Upload masks.zip and run this script
"""

import cv2
import numpy as np
from PIL import Image
import os
import zipfile
from tqdm import tqdm
import matplotlib.pyplot as plt

print("=" * 60)
print("SIMPLE MASK SOFTENING - GAUSSIAN BLUR")
print("No inversions, no transformations")
print("=" * 60)
print()

# ============================================
# CONFIGURATION
# ============================================

MASKS_ZIP = "vessel_masks.zip"
OUTPUT_ZIP = "masks_softened.zip"

# Simple blur parameters
BLUR_KERNEL_SIZE = 5    # Size of blur kernel (3, 5, 7, or 9)
BLUR_SIGMA = 1.5        # Blur strength

# ============================================
# SIMPLE SOFTENING FUNCTION
# ============================================

def soften_mask_simple(mask):
    """
    Simple Gaussian blur - no tricks, no transformations
    Just softens the edges
    """
    # Convert to grayscale if needed
    if len(mask.shape) == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian blur - that's it!
    softened = cv2.GaussianBlur(mask, (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE), BLUR_SIGMA)

    # Convert to RGB (Pix2PixHD expects 3-channel)
    softened_rgb = cv2.cvtColor(softened, cv2.COLOR_GRAY2RGB)

    return softened_rgb

# ============================================
# MAIN PROCESSING
# ============================================

# Step 1: Extract masks.zip
print("Step 1: Extracting masks.zip...")
extract_dir = "temp_masks"
os.makedirs(extract_dir, exist_ok=True)

if not os.path.exists(MASKS_ZIP):
    print(f"❌ Error: {MASKS_ZIP} not found!")
    print("Please upload masks.zip to this directory")
    exit()

with zipfile.ZipFile(MASKS_ZIP, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print(f"✓ Extracted")

# Handle nested 'masks' folder
nested_masks_dir = os.path.join(extract_dir, 'masks')
if os.path.exists(nested_masks_dir) and os.path.isdir(nested_masks_dir):
    print(f"✓ Found nested 'masks/' folder")
    search_dir = nested_masks_dir
else:
    search_dir = extract_dir

print()

# Step 2: Find all mask files
print("Step 2: Finding mask files...")
mask_files = []
for root, dirs, files in os.walk(search_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            if not file.startswith('.') and not file.startswith('__'):
                mask_files.append(os.path.join(root, file))

mask_files = sorted(mask_files)
print(f"✓ Found {len(mask_files)} mask files")
print()

if len(mask_files) == 0:
    print("❌ No image files found!")
    exit()

# Step 3: Process masks
print("Step 3: Applying Gaussian blur...")
output_dir = "softened_masks"
os.makedirs(output_dir, exist_ok=True)

for mask_path in tqdm(mask_files, desc="Processing"):
    # Read mask
    mask = cv2.imread(mask_path)

    if mask is None:
        continue

    # Apply simple blur
    softened = soften_mask_simple(mask)

    # Preserve original filename
    original_filename = os.path.basename(mask_path)
    name_without_ext = os.path.splitext(original_filename)[0]
    output_filename = f"{name_without_ext}.png"

    # Save
    output_path = os.path.join(output_dir, output_filename)
    cv2.imwrite(output_path, softened)

print(f"✓ Processed {len(mask_files)} masks")
print()

# Step 4: Create preview
print("Step 4: Creating preview...")
preview_dir = "preview"
os.makedirs(preview_dir, exist_ok=True)

num_preview = min(3, len(mask_files))
for i in range(num_preview):
    original = cv2.imread(mask_files[i], cv2.IMREAD_GRAYSCALE)

    original_filename = os.path.basename(mask_files[i])
    name_without_ext = os.path.splitext(original_filename)[0]
    softened_path = os.path.join(output_dir, f"{name_without_ext}.png")
    softened = cv2.imread(softened_path, cv2.IMREAD_GRAYSCALE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'Original\n{original_filename}', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(softened, cmap='gray', vmin=0, vmax=255)
    axes[1].set_title(f'Softened (Gaussian Blur)\n{name_without_ext}.png', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.suptitle(f'Simple Gaussian Blur - No Transformations', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{preview_dir}/comparison_{i:03d}.png', dpi=150, bbox_inches='tight')
    plt.close()

print(f"✓ Created {num_preview} previews")
print()

# Step 5: Create output zip
print("Step 5: Creating masks_softened.zip...")
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, file)

print(f"✓ Created {OUTPUT_ZIP}")
print()

# Cleanup
import shutil
shutil.rmtree(extract_dir)

# Final summary
print("=" * 60)
print("SIMPLE SOFTENING COMPLETE!")
print("=" * 60)
print(f"✓ Processed: {len(mask_files)} masks")
print(f"✓ Method: Gaussian Blur ONLY")
print(f"✓ Blur kernel: {BLUR_KERNEL_SIZE}x{BLUR_KERNEL_SIZE}")
print(f"✓ Blur sigma: {BLUR_SIGMA}")
print()
print("WHAT WAS DONE:")
print("  - Simple Gaussian blur applied")
print("  - NO inversions")
print("  - NO transformations")
print("  - NO denoising")
print("  - Masks look exactly the same, just softer edges")
print()
print("NEXT STEPS:")
print("1. Check preview/ - should look like original but blurred")
print("2. If too blurry: reduce BLUR_KERNEL_SIZE to 3")
print("3. If not enough: increase BLUR_KERNEL_SIZE to 7")
print("4. Use masks_softened.zip for training")
print("=" * 60)

SIMPLE MASK SOFTENING - GAUSSIAN BLUR
No inversions, no transformations

Step 1: Extracting masks.zip...
✓ Extracted

Step 2: Finding mask files...
✓ Found 150 mask files

Step 3: Applying Gaussian blur...


Processing: 100%|██████████| 150/150 [00:27<00:00,  5.50it/s]


✓ Processed 150 masks

Step 4: Creating preview...
✓ Created 3 previews

Step 5: Creating masks_softened.zip...
✓ Created masks_softened.zip

SIMPLE SOFTENING COMPLETE!
✓ Processed: 150 masks
✓ Method: Gaussian Blur ONLY
✓ Blur kernel: 5x5
✓ Blur sigma: 1.5

WHAT WAS DONE:
  - Simple Gaussian blur applied
  - NO inversions
  - NO transformations
  - NO denoising
  - Masks look exactly the same, just softer edges

NEXT STEPS:
1. Check preview/ - should look like original but blurred
2. If too blurry: reduce BLUR_KERNEL_SIZE to 3
3. If not enough: increase BLUR_KERNEL_SIZE to 7
4. Use masks_softened.zip for training


In [2]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install opencv-python-headless numpy Pillow tqdm


# ============================================================
# CELL 2: Unzip images.zip (already uploaded in Colab)
# ============================================================
import zipfile
import os

# Path to your already-uploaded images.zip
zip_path = '/content/images.zip'

if not os.path.exists(zip_path):
    print(f"❌ File not found: {zip_path}")
    print("   Make sure images.zip is uploaded to /content/ in Colab")
else:
    print(f"📦 Found: {zip_path}")
    print(f"   Size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

    # Unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/uploaded_data/')
    print("✅ Unzipped successfully!")

    # Auto-detect the images folder
    # Your structure: images.zip -> images/ -> *.jpg/png
    image_dir = None
    for root, dirs, file_list in os.walk('/content/uploaded_data/'):
        # Skip __MACOSX and hidden directories
        if '__MACOSX' in root or '/.' in root:
            continue
        image_files_in_dir = [f for f in file_list
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                              and not f.startswith('.')]
        if len(image_files_in_dir) > 0:
            image_dir = root
            break

    if image_dir:
        all_images = [f for f in os.listdir(image_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                      and not f.startswith('.')]
        print(f"✅ Found {len(all_images)} images in: {image_dir}")
        print(f"📸 Sample files: {all_images[:5]}")
    else:
        print("❌ No images found! Check your zip structure.")
        print("Expected: images.zip -> images/ -> image1.jpg, image2.jpg, ...")
        # List what was extracted
        print("\nExtracted contents:")
        for root, dirs, files in os.walk('/content/uploaded_data/'):
            level = root.replace('/content/uploaded_data/', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for file in files[:10]:
                print(f"{subindent}{file}")
            if len(files) > 10:
                print(f"{subindent}... and {len(files)-10} more files")


# ============================================================
# CELL 3: Vessel Extraction with Step-by-Step Preview
# Upload ONE fundus image and see every processing stage
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files
from tqdm import tqdm

# ── Upload a single image ────────────────────────────────────
print("📂 Upload one fundus image to preview all processing steps:")
uploaded_preview = files.upload()
preview_path = list(uploaded_preview.keys())[0]
print(f"✅ Loaded: {preview_path}")


# ── Helper: show an image inline ────────────────────────────
def show_step(ax, img, title, cmap='gray'):
    """Display a processing step on a matplotlib axis."""
    ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.axis('off')


# ── Remove small connected components ───────────────────────
def _remove_small_components(binary, min_size):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    result = np.zeros_like(binary)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            result[labels == i] = 255
    return result


# ═══════════════════════════════════════════════════════════
#  SIMPLE METHOD — step-by-step preview
# ═══════════════════════════════════════════════════════════
def preview_simple(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ── All processing steps ──────────────────────────────
    # Step 1: Green channel
    green = img[:, :, 1]

    # Step 2: FOV mask
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    # Step 3: CLAHE
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Step 4: Invert
    inverted = cv2.bitwise_not(enhanced)

    # Step 5: Top-hat transform
    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, kernel_tophat)

    # Step 6: Second CLAHE
    clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    enhanced2 = clahe2.apply(tophat)

    # Step 7: Morphological opening (denoise)
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # Step 8: Adaptive thresholding
    block_size = max(11, min(31, (min(w, h) // 40) | 1))
    if block_size % 2 == 0:
        block_size += 1
    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=block_size,
        C=3
    )

    # Step 9: Morphological closing
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Step 10: Remove small components
    min_size = max(15, int(w * h * 0.00005))
    cleaned = _remove_small_components(closed, min_size)

    # Step 11: Final morphological opening
    kernel_final = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_final, iterations=1)

    # Step 12: Apply FOV + strict binary
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    # ── Plot all steps ────────────────────────────────────
    steps = [
        (img_rgb,    'Step 0: Original Fundus',        None),
        (green,      'Step 1: Green Channel',           'gray'),
        (fov_mask,   'Step 2: FOV Mask',                'gray'),
        (enhanced,   'Step 3: CLAHE Enhanced',          'gray'),
        (inverted,   'Step 4: Inverted',                'gray'),
        (tophat,     'Step 5: Top-hat Transform',       'gray'),
        (enhanced2,  'Step 6: CLAHE on Top-hat',        'gray'),
        (opened,     'Step 7: Morph Opening (denoise)', 'gray'),
        (binary,     'Step 8: Adaptive Threshold',      'gray'),
        (closed,     'Step 9: Morph Closing',           'gray'),
        (cleaned,    'Step 10: Remove Small Components','gray'),
        (result,     '✅ FINAL: White Vessels / Black BG','gray'),
    ]

    cols = 3
    rows = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4.5))
    axes = axes.flatten()

    for i, (img_s, title, cmap) in enumerate(steps):
        show_step(axes[i], img_s, title, cmap=cmap)

    # Hide any unused axes
    for j in range(len(steps), len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        'SIMPLE METHOD — Step-by-Step Vessel Extraction',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Final mask unique pixel values: {np.unique(result)}")
    print(f"   {'Binary confirmed! Only 0 and 255.' if set(np.unique(result)).issubset({0,255}) else 'WARNING: Not strictly binary!'}")
    return result


# ═══════════════════════════════════════════════════════════
#  ADVANCED METHOD — step-by-step preview
# ═══════════════════════════════════════════════════════════
def preview_advanced(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Green channel
    green = img[:, :, 1]

    # Step 2: FOV mask
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    # Step 3: CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Step 4: Matched filter bank (12 orientations)
    ksize = 15
    num_orientations = 12
    center = ksize // 2
    responses = []
    print("⏳ Computing matched filter bank (12 orientations)...")
    for oi in tqdm(range(num_orientations), desc="Orientations"):
        angle = (oi * 180.0) / num_orientations
        angle_rad = np.radians(angle)
        kernel = np.zeros((ksize, ksize), dtype=np.float64)
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        sigma = 1.5
        for y in range(ksize):
            for x in range(ksize):
                dx, dy = x - center, y - center
                along = dx * cos_a + dy * sin_a
                perp  = -dx * sin_a + dy * cos_a
                if abs(along) <= center:
                    kernel[y, x] = np.exp(-(perp**2) / (2 * sigma**2))
        k_sum = kernel.sum()
        if k_sum > 0:
            kernel /= k_sum
        kernel -= kernel.mean()
        responses.append(cv2.filter2D(enhanced.astype(np.float64), -1, kernel))

    # Step 5: Max response across orientations
    vessel_response = np.max(responses, axis=0)

    # Step 6: Normalize within FOV
    fov_pixels = vessel_response[fov_mask > 0]
    min_r, max_r = fov_pixels.min(), fov_pixels.max()
    range_r = max_r - min_r if max_r > min_r else 1
    normalized = np.clip(((vessel_response - min_r) / range_r) * 255, 0, 255).astype(np.uint8)
    normalized[fov_mask == 0] = 0

    # Step 7: Otsu threshold
    _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 8: Morph opening
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # Step 9: Morph closing
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Step 10: Remove small components
    min_size = max(20, int(w * h * 0.00008))
    cleaned = _remove_small_components(closed, min_size)

    # Step 11: Apply FOV + strict binary
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    # ── Normalize vessel_response to uint8 for display ──
    vr_display = np.clip(((vessel_response - vessel_response.min()) /
                          (vessel_response.max() - vessel_response.min() + 1e-6)) * 255, 0, 255).astype(np.uint8)

    steps = [
        (img_rgb,      'Step 0: Original Fundus',          None),
        (green,        'Step 1: Green Channel',             'gray'),
        (fov_mask,     'Step 2: FOV Mask',                  'gray'),
        (enhanced,     'Step 3: CLAHE Enhanced',            'gray'),
        (vr_display,   'Step 4: Max Filter Response',       'hot'),
        (normalized,   'Step 5: Normalized (within FOV)',   'gray'),
        (binary,       'Step 6: Otsu Threshold',            'gray'),
        (opened,       'Step 7: Morph Opening',             'gray'),
        (closed,       'Step 8: Morph Closing',             'gray'),
        (cleaned,      'Step 9: Remove Small Components',   'gray'),
        (result,       '✅ FINAL: White Vessels / Black BG', 'gray'),
    ]

    cols = 3
    rows = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4.5))
    axes = axes.flatten()

    for i, (img_s, title, cmap) in enumerate(steps):
        show_step(axes[i], img_s, title, cmap=cmap)

    for j in range(len(steps), len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        'ADVANCED METHOD — Step-by-Step Vessel Extraction',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Final mask unique pixel values: {np.unique(result)}")
    print(f"   {'Binary confirmed! Only 0 and 255.' if set(np.unique(result)).issubset({0,255}) else 'WARNING: Not strictly binary!'}")
    return result


# ── Run preview on uploaded image ───────────────────────────
print("\n" + "="*60)
print("  Running SIMPLE method step-by-step preview...")
print("="*60)
result_simple = preview_simple(preview_path)

# Save the simple result
cv2.imwrite('/content/preview_vessel_mask_simple.png', result_simple)
print("\n💾 Saved: /content/preview_vessel_mask_simple.png")


# ── Also run ADVANCED and compare ───────────────────────────
print("\n" + "="*60)
print("  Running ADVANCED method step-by-step preview...")
print("="*60)
result_advanced = preview_advanced(preview_path)

cv2.imwrite('/content/preview_vessel_mask_advanced.png', result_advanced)
print("\n💾 Saved: /content/preview_vessel_mask_advanced.png")


# ── Side-by-side final comparison ───────────────────────────
img_orig = cv2.imread(preview_path)
img_rgb  = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_step(axes[0], img_rgb,          'Original Fundus',          None)
show_step(axes[1], result_simple,    'Simple Method (Final)',     'gray')
show_step(axes[2], result_advanced,  'Advanced Method (Final)',   'gray')
fig.suptitle('Final Comparison: Simple vs Advanced', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Done! Choose the method that looks better for your images.")
print("   Then use that method in CELL 4 (METHOD = 'simple' or 'advanced')")


# ── Keep extract_vessels() for batch processing in Cell 4 ───
def extract_vessels(image_path, method='simple'):
    img = cv2.imread(image_path)
    if img is None:
        print(f"  ⚠️ Failed to load: {image_path}")
        return None
    h, w = img.shape[:2]
    green = img[:, :, 1]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)
    if method == 'advanced':
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        enhanced = clahe.apply(green)
        ksize, num_orientations, center = 15, 12, 7
        responses = []
        for oi in range(num_orientations):
            angle_rad = np.radians((oi * 180.0) / num_orientations)
            kernel = np.zeros((ksize, ksize), dtype=np.float64)
            cos_a, sin_a, sigma = np.cos(angle_rad), np.sin(angle_rad), 1.5
            for y in range(ksize):
                for x in range(ksize):
                    dx, dy = x - center, y - center
                    if abs(dx * cos_a + dy * sin_a) <= center:
                        kernel[y, x] = np.exp(-((-dx * sin_a + dy * cos_a)**2) / (2 * sigma**2))
            k_sum = kernel.sum()
            if k_sum > 0: kernel /= k_sum
            kernel -= kernel.mean()
            responses.append(cv2.filter2D(enhanced.astype(np.float64), -1, kernel))
        vessel_response = np.max(responses, axis=0)
        fov_pixels = vessel_response[fov_mask > 0]
        min_r, max_r = fov_pixels.min(), fov_pixels.max()
        normalized = np.clip(((vessel_response - min_r) / (max_r - min_r + 1e-6)) * 255, 0, 255).astype(np.uint8)
        normalized[fov_mask == 0] = 0
        _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_o, iterations=1)
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, k_c, iterations=1)
        cleaned = _remove_small_components(cleaned, max(20, int(w * h * 0.00008)))
    else:
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(green)
        inverted = cv2.bitwise_not(enhanced)
        k_th = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, k_th)
        clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        enhanced2 = clahe2.apply(tophat)
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, k_o, iterations=1)
        bs = max(11, min(31, (min(w, h) // 40) | 1))
        if bs % 2 == 0: bs += 1
        binary = cv2.adaptiveThreshold(opened, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, bs, 3)
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_c, iterations=1)
        cleaned = _remove_small_components(closed, max(15, int(w * h * 0.00005)))
        k_f = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, k_f, iterations=1)
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)
    return result


# ============================================================
# CELL 4: Process All Images & Create Vessel Masks
# ============================================================
import glob

# Output directory for vessel masks
output_dir = '/content/vessel_masks/'
os.makedirs(output_dir, exist_ok=True)

# Choose method: 'simple' (fast) or 'advanced' (better quality, slower)
METHOD = 'simple'  # <-- Change to 'advanced' if needed

print(f"🔬 Method: {METHOD}")
print(f"📂 Input:  {image_dir}")
print(f"📂 Output: {output_dir}")
print("=" * 60)

# Get all image files
image_files = sorted([
    os.path.join(image_dir, f) for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
    and not f.startswith('.')
])

print(f"🖼️  Found {len(image_files)} images to process\n")

success_count = 0
fail_count = 0

for img_path in tqdm(image_files, desc="Extracting vessels"):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    output_path = os.path.join(output_dir, f"{name_only}.png")

    mask = extract_vessels(img_path, method=METHOD)

    if mask is not None:
        cv2.imwrite(output_path, mask)
        success_count += 1
    else:
        fail_count += 1

print(f"\n{'=' * 60}")
print(f"✅ Successfully processed: {success_count}/{len(image_files)}")
if fail_count > 0:
    print(f"❌ Failed: {fail_count}")
print(f"📁 Vessel masks saved to: {output_dir}")
print(f"{'=' * 60}")


# ============================================================
# CELL 5: Preview Some Results (Side by Side)
# ============================================================
import matplotlib.pyplot as plt

# Show up to 5 random samples
sample_files = image_files[:5]  # First 5 images

fig, axes = plt.subplots(len(sample_files), 2, figsize=(12, 5 * len(sample_files)))

if len(sample_files) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_files):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    mask_path = os.path.join(output_dir, f"{name_only}.png")

    # Load original
    original = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    # Load mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Plot
    axes[idx][0].imshow(original_rgb)
    axes[idx][0].set_title(f'Original: {filename}', fontsize=10)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(mask, cmap='gray', vmin=0, vmax=255)
    axes[idx][1].set_title(f'Vessel Mask: {name_only}.png', fontsize=10)
    axes[idx][1].axis('off')

plt.suptitle('Fundus Images → Vessel Masks (White on Black)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show pixel value stats to confirm binary
sample_mask = cv2.imread(os.path.join(output_dir, os.listdir(output_dir)[0]), cv2.IMREAD_GRAYSCALE)
unique_vals = np.unique(sample_mask)
print(f"\n🔍 Pixel value check: unique values = {unique_vals}")
print(f"   ✅ {'Binary confirmed! Only 0 and 255.' if set(unique_vals).issubset({0, 255}) else '⚠️ Not strictly binary!'}")


# ============================================================
# CELL 6: Zip & Download Vessel Masks
# ============================================================
import shutil

# Create zip of vessel masks
zip_output = '/content/vessel_masks'
shutil.make_archive(zip_output, 'zip', '/content/', 'vessel_masks')

print(f"📦 Created: vessel_masks.zip")
print(f"   Contains {len(os.listdir(output_dir))} vessel mask images")
print(f"   Format: White vessels (255) on black background (0)")
print()

# Auto-download
from google.colab import files
files.download('/content/vessel_masks.zip')
print("⬇️  Download started!")


# ============================================================
# CELL 7 (OPTIONAL): Re-run with ADVANCED method if needed
# ============================================================

# Uncomment all lines below to re-run with advanced matched filtering:

# output_dir_adv = '/content/vessel_masks_advanced/'
# os.makedirs(output_dir_adv, exist_ok=True)
#
# print("🔬 Re-running with ADVANCED method...")
# print("=" * 60)
#
# for img_path in tqdm(image_files, desc="Advanced extraction"):
#     filename = os.path.basename(img_path)
#     name_only = os.path.splitext(filename)[0]
#     output_path = os.path.join(output_dir_adv, f"{name_only}.png")
#     mask = extract_vessels(img_path, method='advanced')
#     if mask is not None:
#         cv2.imwrite(output_path, mask)
#
# shutil.make_archive('/content/vessel_masks_advanced', 'zip', '/content/', 'vessel_masks_advanced')
# files.download('/content/vessel_masks_advanced.zip')
# print("⬇️  Advanced masks downloaded!")

Output hidden; open in https://colab.research.google.com to view.

In [4]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install opencv-python-headless numpy Pillow tqdm


# ============================================================
# CELL 2: Unzip images.zip (already uploaded in Colab)
# ============================================================
import zipfile
import os

# Path to your already-uploaded images.zip
zip_path = '/content/images.zip'

if not os.path.exists(zip_path):
    print(f"❌ File not found: {zip_path}")
    print("   Make sure images.zip is uploaded to /content/ in Colab")
else:
    print(f"📦 Found: {zip_path}")
    print(f"   Size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

    # Unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/uploaded_data/')
    print("✅ Unzipped successfully!")

    # Auto-detect the images folder
    # Your structure: images.zip -> images/ -> *.jpg/png
    image_dir = None
    for root, dirs, file_list in os.walk('/content/uploaded_data/'):
        # Skip __MACOSX and hidden directories
        if '__MACOSX' in root or '/.' in root:
            continue
        image_files_in_dir = [f for f in file_list
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                              and not f.startswith('.')]
        if len(image_files_in_dir) > 0:
            image_dir = root
            break

    if image_dir:
        all_images = [f for f in os.listdir(image_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
                      and not f.startswith('.')]
        print(f"✅ Found {len(all_images)} images in: {image_dir}")
        print(f"📸 Sample files: {all_images[:5]}")
    else:
        print("❌ No images found! Check your zip structure.")
        print("Expected: images.zip -> images/ -> image1.jpg, image2.jpg, ...")
        # List what was extracted
        print("\nExtracted contents:")
        for root, dirs, files in os.walk('/content/uploaded_data/'):
            level = root.replace('/content/uploaded_data/', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            subindent = ' ' * 2 * (level + 1)
            for file in files[:10]:
                print(f"{subindent}{file}")
            if len(files) > 10:
                print(f"{subindent}... and {len(files)-10} more files")


# ============================================================
# CELL 3: Vessel Extraction with Step-by-Step Preview
# Upload ONE fundus image and see every processing stage
# ============================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files
from tqdm import tqdm

# ── Upload a single image ────────────────────────────────────
print("📂 Upload one fundus image to preview all processing steps:")
uploaded_preview = files.upload()
preview_path = list(uploaded_preview.keys())[0]
print(f"✅ Loaded: {preview_path}")


# ── Helper: show an image inline ────────────────────────────
def show_step(ax, img, title, cmap='gray'):
    """Display a processing step on a matplotlib axis."""
    ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.axis('off')


# ── Remove small connected components ───────────────────────
def _remove_small_components(binary, min_size):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    result = np.zeros_like(binary)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            result[labels == i] = 255
    return result


# ═══════════════════════════════════════════════════════════
#  SIMPLE METHOD — step-by-step preview
# ═══════════════════════════════════════════════════════════
def preview_simple(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ── All processing steps ──────────────────────────────
    # Step 1: Green channel
    green = img[:, :, 1]

    # Step 2: FOV mask
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    # Step 3: CLAHE
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Step 4: Invert
    inverted = cv2.bitwise_not(enhanced)

    # Step 5: Top-hat transform
    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, kernel_tophat)

    # Step 6: Second CLAHE
    clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    enhanced2 = clahe2.apply(tophat)

    # Step 7: Morphological opening (denoise)
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # Step 8: Adaptive thresholding
    block_size = max(11, min(31, (min(w, h) // 40) | 1))
    if block_size % 2 == 0:
        block_size += 1
    binary = cv2.adaptiveThreshold(
        opened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=block_size,
        C=3
    )

    # Step 9: Morphological closing
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Step 10: Remove small components
    min_size = max(15, int(w * h * 0.00005))
    cleaned = _remove_small_components(closed, min_size)

    # Step 11: Final morphological opening
    kernel_final = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_final, iterations=1)

    # Step 12: Apply FOV + strict binary
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    # ── Plot all steps ────────────────────────────────────
    steps = [
        (img_rgb,    'Step 0: Original Fundus',        None),
        (green,      'Step 1: Green Channel',           'gray'),
        (fov_mask,   'Step 2: FOV Mask',                'gray'),
        (enhanced,   'Step 3: CLAHE Enhanced',          'gray'),
        (inverted,   'Step 4: Inverted',                'gray'),
        (tophat,     'Step 5: Top-hat Transform',       'gray'),
        (enhanced2,  'Step 6: CLAHE on Top-hat',        'gray'),
        (opened,     'Step 7: Morph Opening (denoise)', 'gray'),
        (binary,     'Step 8: Adaptive Threshold',      'gray'),
        (closed,     'Step 9: Morph Closing',           'gray'),
        (cleaned,    'Step 10: Remove Small Components','gray'),
        (result,     '✅ FINAL: White Vessels / Black BG','gray'),
    ]

    cols = 3
    rows = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4.5))
    axes = axes.flatten()

    for i, (img_s, title, cmap) in enumerate(steps):
        show_step(axes[i], img_s, title, cmap=cmap)

    # Hide any unused axes
    for j in range(len(steps), len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        'SIMPLE METHOD — Step-by-Step Vessel Extraction',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Final mask unique pixel values: {np.unique(result)}")
    print(f"   {'Binary confirmed! Only 0 and 255.' if set(np.unique(result)).issubset({0,255}) else 'WARNING: Not strictly binary!'}")
    return result


# ═══════════════════════════════════════════════════════════
#  ADVANCED METHOD — step-by-step preview
# ═══════════════════════════════════════════════════════════
def preview_advanced(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Green channel
    green = img[:, :, 1]

    # Step 2: FOV mask
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)

    # Step 3: CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    # Step 4: Matched filter bank (12 orientations)
    ksize = 15
    num_orientations = 12
    center = ksize // 2
    responses = []
    print("⏳ Computing matched filter bank (12 orientations)...")
    for oi in tqdm(range(num_orientations), desc="Orientations"):
        angle = (oi * 180.0) / num_orientations
        angle_rad = np.radians(angle)
        kernel = np.zeros((ksize, ksize), dtype=np.float64)
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        sigma = 1.5
        for y in range(ksize):
            for x in range(ksize):
                dx, dy = x - center, y - center
                along = dx * cos_a + dy * sin_a
                perp  = -dx * sin_a + dy * cos_a
                if abs(along) <= center:
                    kernel[y, x] = np.exp(-(perp**2) / (2 * sigma**2))
        k_sum = kernel.sum()
        if k_sum > 0:
            kernel /= k_sum
        kernel -= kernel.mean()
        responses.append(cv2.filter2D(enhanced.astype(np.float64), -1, kernel))

    # Step 5: Max response across orientations
    vessel_response = np.max(responses, axis=0)

    # Step 6: Normalize within FOV
    fov_pixels = vessel_response[fov_mask > 0]
    min_r, max_r = fov_pixels.min(), fov_pixels.max()
    range_r = max_r - min_r if max_r > min_r else 1
    normalized = np.clip(((vessel_response - min_r) / range_r) * 255, 0, 255).astype(np.uint8)
    normalized[fov_mask == 0] = 0

    # Step 7: Otsu threshold
    _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 8: Morph opening
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open, iterations=1)

    # Step 9: Morph closing
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel_close, iterations=1)

    # Step 10: Remove small components
    min_size = max(20, int(w * h * 0.00008))
    cleaned = _remove_small_components(closed, min_size)

    # Step 11: Apply FOV + strict binary
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)

    # ── Normalize vessel_response to uint8 for display ──
    vr_display = np.clip(((vessel_response - vessel_response.min()) /
                          (vessel_response.max() - vessel_response.min() + 1e-6)) * 255, 0, 255).astype(np.uint8)

    steps = [
        (img_rgb,      'Step 0: Original Fundus',          None),
        (green,        'Step 1: Green Channel',             'gray'),
        (fov_mask,     'Step 2: FOV Mask',                  'gray'),
        (enhanced,     'Step 3: CLAHE Enhanced',            'gray'),
        (vr_display,   'Step 4: Max Filter Response',       'hot'),
        (normalized,   'Step 5: Normalized (within FOV)',   'gray'),
        (binary,       'Step 6: Otsu Threshold',            'gray'),
        (opened,       'Step 7: Morph Opening',             'gray'),
        (closed,       'Step 8: Morph Closing',             'gray'),
        (cleaned,      'Step 9: Remove Small Components',   'gray'),
        (result,       '✅ FINAL: White Vessels / Black BG', 'gray'),
    ]

    cols = 3
    rows = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4.5))
    axes = axes.flatten()

    for i, (img_s, title, cmap) in enumerate(steps):
        show_step(axes[i], img_s, title, cmap=cmap)

    for j in range(len(steps), len(axes)):
        axes[j].axis('off')

    fig.suptitle(
        'ADVANCED METHOD — Step-by-Step Vessel Extraction',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    print(f"\n✅ Final mask unique pixel values: {np.unique(result)}")
    print(f"   {'Binary confirmed! Only 0 and 255.' if set(np.unique(result)).issubset({0,255}) else 'WARNING: Not strictly binary!'}")
    return result


# ── Run preview on uploaded image ───────────────────────────
print("\n" + "="*60)
print("  Running SIMPLE method step-by-step preview...")
print("="*60)
result_simple = preview_simple(preview_path)

# Save the simple result
cv2.imwrite('/content/preview_vessel_mask_simple.png', result_simple)
print("\n💾 Saved: /content/preview_vessel_mask_simple.png")


# ── Also run ADVANCED and compare ───────────────────────────
print("\n" + "="*60)
print("  Running ADVANCED method step-by-step preview...")
print("="*60)
result_advanced = preview_advanced(preview_path)

cv2.imwrite('/content/preview_vessel_mask_advanced.png', result_advanced)
print("\n💾 Saved: /content/preview_vessel_mask_advanced.png")


# ── Side-by-side final comparison ───────────────────────────
img_orig = cv2.imread(preview_path)
img_rgb  = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_step(axes[0], img_rgb,          'Original Fundus',          None)
show_step(axes[1], result_simple,    'Simple Method (Final)',     'gray')
show_step(axes[2], result_advanced,  'Advanced Method (Final)',   'gray')
fig.suptitle('Final Comparison: Simple vs Advanced', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Done! Choose the method that looks better for your images.")
print("   Then use that method in CELL 4 (METHOD = 'simple' or 'advanced')")


# ── Keep extract_vessels() for batch processing in Cell 4 ───
def extract_vessels(image_path, method='simple'):
    img = cv2.imread(image_path)
    if img is None:
        print(f"  ⚠️ Failed to load: {image_path}")
        return None
    h, w = img.shape[:2]
    green = img[:, :, 1]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred_fov = cv2.GaussianBlur(gray, (15, 15), 0)
    _, fov_mask = cv2.threshold(blurred_fov, 15, 255, cv2.THRESH_BINARY)
    fov_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_mask = cv2.erode(fov_mask, fov_kernel, iterations=1)
    if method == 'advanced':
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
        enhanced = clahe.apply(green)
        ksize, num_orientations, center = 15, 12, 7
        responses = []
        for oi in range(num_orientations):
            angle_rad = np.radians((oi * 180.0) / num_orientations)
            kernel = np.zeros((ksize, ksize), dtype=np.float64)
            cos_a, sin_a, sigma = np.cos(angle_rad), np.sin(angle_rad), 1.5
            for y in range(ksize):
                for x in range(ksize):
                    dx, dy = x - center, y - center
                    if abs(dx * cos_a + dy * sin_a) <= center:
                        kernel[y, x] = np.exp(-((-dx * sin_a + dy * cos_a)**2) / (2 * sigma**2))
            k_sum = kernel.sum()
            if k_sum > 0: kernel /= k_sum
            kernel -= kernel.mean()
            responses.append(cv2.filter2D(enhanced.astype(np.float64), -1, kernel))
        vessel_response = np.max(responses, axis=0)
        fov_pixels = vessel_response[fov_mask > 0]
        min_r, max_r = fov_pixels.min(), fov_pixels.max()
        normalized = np.clip(((vessel_response - min_r) / (max_r - min_r + 1e-6)) * 255, 0, 255).astype(np.uint8)
        normalized[fov_mask == 0] = 0
        _, binary = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k_o, iterations=1)
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, k_c, iterations=1)
        cleaned = _remove_small_components(cleaned, max(20, int(w * h * 0.00008)))
    else:
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(green)
        inverted = cv2.bitwise_not(enhanced)
        k_th = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        tophat = cv2.morphologyEx(inverted, cv2.MORPH_TOPHAT, k_th)
        clahe2 = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        enhanced2 = clahe2.apply(tophat)
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        opened = cv2.morphologyEx(enhanced2, cv2.MORPH_OPEN, k_o, iterations=1)
        bs = max(11, min(31, (min(w, h) // 40) | 1))
        if bs % 2 == 0: bs += 1
        binary = cv2.adaptiveThreshold(opened, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, bs, 3)
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_c, iterations=1)
        cleaned = _remove_small_components(closed, max(15, int(w * h * 0.00005)))
        k_f = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, k_f, iterations=1)
    cleaned[fov_mask == 0] = 0
    _, result = cv2.threshold(cleaned, 128, 255, cv2.THRESH_BINARY)
    return result


# ============================================================
# CELL 4: Process All Images & Create Vessel Masks
# ============================================================
import glob

# Output directory for vessel masks
output_dir = '/content/vessel_masks/'
os.makedirs(output_dir, exist_ok=True)

# Choose method: 'simple' (fast) or 'advanced' (better quality, slower)
METHOD = 'simple'  # <-- Change to 'advanced' if needed

print(f"🔬 Method: {METHOD}")
print(f"📂 Input:  {image_dir}")
print(f"📂 Output: {output_dir}")
print("=" * 60)

# Get all image files
image_files = sorted([
    os.path.join(image_dir, f) for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))
    and not f.startswith('.')
])

print(f"🖼️  Found {len(image_files)} images to process\n")

success_count = 0
fail_count = 0

for img_path in tqdm(image_files, desc="Extracting vessels"):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    output_path = os.path.join(output_dir, f"{name_only}.png")

    mask = extract_vessels(img_path, method=METHOD)

    if mask is not None:
        cv2.imwrite(output_path, mask)
        success_count += 1
    else:
        fail_count += 1

print(f"\n{'=' * 60}")
print(f"✅ Successfully processed: {success_count}/{len(image_files)}")
if fail_count > 0:
    print(f"❌ Failed: {fail_count}")
print(f"📁 Vessel masks saved to: {output_dir}")
print(f"{'=' * 60}")


# ============================================================
# CELL 5: Preview Some Results (Side by Side)
# ============================================================
import matplotlib.pyplot as plt

# Show up to 5 random samples
sample_files = image_files[:5]  # First 5 images

fig, axes = plt.subplots(len(sample_files), 2, figsize=(12, 5 * len(sample_files)))

if len(sample_files) == 1:
    axes = [axes]

for idx, img_path in enumerate(sample_files):
    filename = os.path.basename(img_path)
    name_only = os.path.splitext(filename)[0]
    mask_path = os.path.join(output_dir, f"{name_only}.png")

    # Load original
    original = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    # Load mask
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Plot
    axes[idx][0].imshow(original_rgb)
    axes[idx][0].set_title(f'Original: {filename}', fontsize=10)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(mask, cmap='gray', vmin=0, vmax=255)
    axes[idx][1].set_title(f'Vessel Mask: {name_only}.png', fontsize=10)
    axes[idx][1].axis('off')

plt.suptitle('Fundus Images → Vessel Masks (White on Black)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show pixel value stats to confirm binary
sample_mask = cv2.imread(os.path.join(output_dir, os.listdir(output_dir)[0]), cv2.IMREAD_GRAYSCALE)
unique_vals = np.unique(sample_mask)
print(f"\n🔍 Pixel value check: unique values = {unique_vals}")
print(f"   ✅ {'Binary confirmed! Only 0 and 255.' if set(unique_vals).issubset({0, 255}) else '⚠️ Not strictly binary!'}")


# ============================================================
# CELL 6: Zip & Download Vessel Masks
# ============================================================
import shutil

# Create zip of vessel masks
zip_output = '/content/vessel_masks'
shutil.make_archive(zip_output, 'zip', '/content/', 'vessel_masks')

print(f"📦 Created: vessel_masks.zip")
print(f"   Contains {len(os.listdir(output_dir))} vessel mask images")
print(f"   Format: White vessels (255) on black background (0)")
print()

# Auto-download
from google.colab import files
files.download('/content/vessel_masks.zip')
print("⬇️  Download started!")


# ============================================================
# CELL 7 (OPTIONAL): Re-run with ADVANCED method if needed
# ============================================================

# Uncomment all lines below to re-run with advanced matched filtering:

# output_dir_adv = '/content/vessel_masks_advanced/'
# os.makedirs(output_dir_adv, exist_ok=True)
#
# print("🔬 Re-running with ADVANCED method...")
# print("=" * 60)
#
# for img_path in tqdm(image_files, desc="Advanced extraction"):
#     filename = os.path.basename(img_path)
#     name_only = os.path.splitext(filename)[0]
#     output_path = os.path.join(output_dir_adv, f"{name_only}.png")
#     mask = extract_vessels(img_path, method='advanced')
#     if mask is not None:
#         cv2.imwrite(output_path, mask)
#
# shutil.make_archive('/content/vessel_masks_advanced', 'zip', '/content/', 'vessel_masks_advanced')
# files.download('/content/vessel_masks_advanced.zip')
# print("⬇️  Advanced masks downloaded!")

Output hidden; open in https://colab.research.google.com to view.